# IAQF 2026 — Reproducible Code Addendum

## Stablecoin Pricing Dynamics and Cross-Currency Basis During the SVB Crisis

This notebook contains **all code** used to produce the figures, tables, and statistical results
reported in them IAQF Final submission.

### Instructions

1. **Requirements**: Install dependencies with `pip install -r requirements.txt`
2. **Data collection** (Section 1): This section fetches raw data from Binance, Coinbase, and Kraken APIs.
   It is included for completeness but **should NOT be re-run** — the raw data is already cached in `data_raw/`.
   Running it requires active internet and may take several hours due to API rate limits.
3. **All other sections** can be run sequentially to reproduce every result in the paper.
4. **Output directories**: Processed data → `data_processed/`, Tables → `tables/`, Column figures → `figures_col/`

### Study Window
- **Period**: March 1–21, 2023 (1-minute frequency)
- **Regimes**: Pre-SVB (Mar 1–10), Crisis (Mar 10–13), Post-SVB (Mar 13–21)
- **Exchanges**: Binance, Coinbase, Kraken
- **Assets**: BTC/USD, BTC/USDT, BTC/USDC, stablecoin/USD pairs


## Requirements

```
pandas>=2.0.0
numpy>=1.24.0
matplotlib>=3.7.0
seaborn>=0.13.0
requests>=2.28.0
statsmodels>=0.14.0
scipy>=1.10.0
pyarrow>=14.0.0
```


## Common Imports and Path Setup


In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen, select_order
from statsmodels.stats.multitest import multipletests
from scipy import stats as sp_stats
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

# Path setup — assumes notebook is run from the project root
ROOT = os.getcwd()
DATA_RAW       = os.path.join(ROOT, 'data_raw')
DATA_PROCESSED = os.path.join(ROOT, 'data_processed')
FIGURES_DIR    = os.path.join(ROOT, 'figures')
FIGURES_COL    = os.path.join(ROOT, 'figures_col')
TABLES_DIR     = os.path.join(ROOT, 'tables')
for d in [DATA_RAW, DATA_PROCESSED, FIGURES_DIR, FIGURES_COL, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Project root: {ROOT}')
print('All output directories created/verified.')


Project root: /Users/dhruvkohli/Desktop/Github Repos/IAQF2026
All output directories created/verified.


---
# Section 1: Data Collection (DO NOT RUN)

**⚠️ DO NOT RUN this cell.** The data has already been collected and is stored in `data_raw/`.
Running this cell would re-fetch ~14 datasets from exchange APIs, which takes several hours
due to rate limiting (especially Kraken's trade-level API).

This code is included for completeness and reproducibility documentation only.

**Source**: `src/01_fetch_data.py`


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  DO NOT RUN THIS CELL — Data already cached in data_raw/       ║
# ║  Uncomment the last line to re-fetch (takes several hours)      ║
# ╚══════════════════════════════════════════════════════════════════╝

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
import time
import os
from io import BytesIO
from zipfile import ZipFile

# Time window
START_DT = datetime(2023, 3, 1, 0, 0, 0, tzinfo=timezone.utc)
END_DT   = datetime(2023, 3, 21, 23, 59, 0, tzinfo=timezone.utc)

START_MS = int(START_DT.timestamp() * 1000)
END_MS   = int(END_DT.timestamp() * 1000)

DATA_DIR = 'data_raw'
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Study window: {START_DT.isoformat()} to {END_DT.isoformat()}")

# ============================================================
# Binance Fetcher
# ============================================================
def fetch_binance_vision(symbol: str, interval: str = '1m', start_ms: int = START_MS, end_ms: int = END_MS) -> pd.DataFrame:
    start_dt_local = pd.Timestamp(start_ms, unit='ms', tz='UTC')
    end_dt_local = pd.Timestamp(end_ms, unit='ms', tz='UTC')
    all_dfs = []
    current = start_dt_local.to_period('M')
    end_period = end_dt_local.to_period('M')

    while current <= end_period:
        year_month = current.strftime('%Y-%m')
        url = f"https://data.binance.vision/data/spot/monthly/klines/{symbol}/{interval}/{symbol}-{interval}-{year_month}.zip"
        print(f"    Downloading {url}")
        resp = requests.get(url, timeout=60)
        if resp.status_code == 404:
            print(f"    WARNING: No data at {url}")
            current += 1
            continue
        resp.raise_for_status()
        with ZipFile(BytesIO(resp.content)) as zf:
            csv_name = zf.namelist()[0]
            df_chunk = pd.read_csv(
                zf.open(csv_name), header=None,
                names=['open_time', 'open', 'high', 'low', 'close', 'volume',
                       'close_time', 'quote_volume', 'trades', 'taker_buy_base',
                       'taker_buy_quote', 'ignore']
            )
            all_dfs.append(df_chunk)
        current += 1

    if not all_dfs:
        return pd.DataFrame()

    df = pd.concat(all_dfs, ignore_index=True)
    df['timestamp'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    for col in ['open', 'high', 'low', 'close', 'volume', 'quote_volume']:
        df[col] = df[col].astype(float)
    df['trades'] = df['trades'].astype(int)
    df = df.set_index('timestamp').sort_index()
    df = df[~df.index.duplicated(keep='first')]
    df = df[(df.index >= pd.Timestamp(start_ms, unit='ms', tz='UTC')) &
            (df.index <= pd.Timestamp(end_ms, unit='ms', tz='UTC'))]
    return df

def fetch_binance_klines(symbol: str, interval: str = '1m', start_ms: int = START_MS, end_ms: int = END_MS, limit: int = 1000) -> pd.DataFrame:
    url = 'https://api.binance.com/api/v3/klines'
    fallback_url = 'https://api.binance.us/api/v3/klines'
    used_fallback = False
    all_data = []
    current_start = start_ms

    while current_start < end_ms:
        params = {'symbol': symbol, 'interval': interval, 'startTime': current_start, 'endTime': end_ms, 'limit': limit}
        resp = requests.get(url, params=params, timeout=30)
        if resp.status_code == 451 and not used_fallback:
            url = fallback_url
            used_fallback = True
            continue
        if resp.status_code == 429:
            time.sleep(60)
            continue
        if resp.status_code == 400 and used_fallback:
            return fetch_binance_vision(symbol, interval, start_ms, end_ms)
        resp.raise_for_status()
        data = resp.json()
        if not data:
            break
        all_data.extend(data)
        current_start = data[-1][6] + 1
        time.sleep(0.15)

    if not all_data:
        return pd.DataFrame()

    df = pd.DataFrame(all_data, columns=['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore'])
    df['timestamp'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = df[col].astype(float)
    df['trades'] = df['trades'].astype(int)
    df = df.set_index('timestamp')
    df = df[~df.index.duplicated(keep='first')]
    return df

# ============================================================
# Coinbase Fetcher
# ============================================================
def fetch_coinbase_candles(product_id: str, granularity: int = 60, start_dt: datetime = START_DT, end_dt: datetime = END_DT) -> pd.DataFrame:
    url = f'https://api.coinbase.com/api/v3/brokerage/market/products/{product_id}/candles'
    gran_str = 'ONE_MINUTE'
    all_data = []
    chunk_seconds = 300 * granularity
    current_start = start_dt

    while current_start < end_dt:
        current_end = min(current_start + timedelta(seconds=chunk_seconds), end_dt)
        params = {'granularity': gran_str, 'start': int(current_start.timestamp()), 'end': int(current_end.timestamp())}
        resp = requests.get(url, params=params, timeout=30)
        if resp.status_code == 429:
            time.sleep(5)
            continue
        if resp.status_code == 404:
            return pd.DataFrame()
        resp.raise_for_status()
        data = resp.json().get('candles', [])
        if data:
            all_data.extend(data)
        current_start = current_end
        time.sleep(0.2)

    if not all_data:
        return pd.DataFrame()

    df = pd.DataFrame(all_data)
    df['timestamp'] = pd.to_datetime(df['start'].astype(int), unit='s', utc=True)
    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = df[col].astype(float)
    df = df.set_index('timestamp').sort_index()
    df = df[~df.index.duplicated(keep='first')]
    return df

# ============================================================
# Kraken Fetcher (Trades to OHLCV)
# ============================================================
def fetch_kraken_ohlcv(pair: str, start_dt: datetime = START_DT, end_dt: datetime = END_DT) -> pd.DataFrame:
    url = 'https://api.kraken.com/0/public/Trades'
    since_ns = int(start_dt.timestamp() * 1e9)
    end_ts = end_dt.timestamp()
    all_trades = []
    call_count = 0

    while True:
        params = {'pair': pair, 'since': str(since_ns)}
        resp = requests.get(url, params=params, timeout=30)
        if resp.status_code == 429:
            time.sleep(10)
            continue
        resp.raise_for_status()
        data = resp.json()
        if data.get('error'):
            err = data['error'][0] if isinstance(data['error'], list) and len(data['error']) > 0 else str(data['error'])
            if 'Rate limit' in err or 'Too many requests' in err:
                print(f"  Kraken Rate Limit: {err}, sleeping 10s...")
                time.sleep(10)
                continue
            else:
                print(f"  Kraken API Error for {pair}: {err}")
                break
        result = data['result']
        new_since = int(result['last'])
        trade_key = [k for k in result if k != 'last'][0]
        trades = result[trade_key]
        if not trades or new_since == since_ns:
            break
        all_trades.extend(trades)
        since_ns = new_since
        call_count += 1
        last_trade_ts = float(trades[-1][2])
        if last_trade_ts >= end_ts:
            break
        if call_count % 14 == 0:
            time.sleep(3)
        else:
            time.sleep(0.1)

    if not all_trades:
        return pd.DataFrame()

    df_trades = pd.DataFrame(all_trades, columns=['price', 'volume', 'time', 'side', 'type', 'misc', 'trade_id'])
    df_trades['price'] = df_trades['price'].astype(float)
    df_trades['volume'] = df_trades['volume'].astype(float)
    df_trades['timestamp'] = pd.to_datetime(df_trades['time'].astype(float), unit='s', utc=True)
    df_trades = df_trades[(df_trades['timestamp'] >= start_dt) & (df_trades['timestamp'] <= end_dt)]
    df_trades = df_trades.set_index('timestamp')
    ohlcv = df_trades['price'].resample('1min').ohlc()
    ohlcv['volume'] = df_trades['volume'].resample('1min').sum()
    ohlcv['trades'] = df_trades['price'].resample('1min').count()
    ohlcv = ohlcv.dropna(subset=['close'])
    return ohlcv

# ============================================================
# Fetch & Cache
# ============================================================
def cache_path(name: str) -> str:
    return os.path.join(DATA_DIR, f"{name}.parquet")

def load_or_fetch(name: str, fetch_fn, *args, **kwargs) -> pd.DataFrame:
    path = cache_path(name)
    if os.path.exists(path):
        print(f"  Loading cached: {name}")
        df = pd.read_parquet(path)
        if not isinstance(df.index, pd.DatetimeIndex):
            if 'timestamp' in df.columns:
                df = df.set_index('timestamp')
        return df

    sparsh_path = os.path.join('data_sparsh', f"{name}.parquet")
    if os.path.exists(sparsh_path):
        print(f"  Copying from data_sparsh: {name}")
        df = pd.read_parquet(sparsh_path)
        if not isinstance(df.index, pd.DatetimeIndex):
            if 'timestamp' in df.columns:
                df = df.set_index('timestamp')
        df.to_parquet(path)
        return df

    print(f"  Fetching: {name} ...")
    df = fetch_fn(*args, **kwargs)
    if not df.empty:
        df.to_parquet(path)
        print(f"  Saved {len(df):,} rows -> {path}")
    return df

def fetch_all_data():
    pairs_conf = [
        # Binance BTC Pairs
        ('binance_btcusdt', fetch_binance_klines, 'BTCUSDT'),
        ('binance_btcusdc', fetch_binance_klines, 'BTCUSDC'),
        ('binance_btceur',  fetch_binance_klines, 'BTCEUR'),
        # Binance Stablecoin Pairs
        ('binance_usdcusdt', fetch_binance_klines, 'USDCUSDT'),

        # Coinbase BTC Pairs
        ('coinbase_btcusd', fetch_coinbase_candles, 'BTC-USD'),
        ('coinbase_btcusdt', fetch_coinbase_candles, 'BTC-USDT'),
        ('coinbase_btceur',  fetch_coinbase_candles, 'BTC-EUR'),
        # Coinbase Stablecoin Pairs
        ('coinbase_usdtusd', fetch_coinbase_candles, 'USDT-USD'),

        # Kraken BTC Pairs
        ('kraken_btcusd',  fetch_kraken_ohlcv, 'XXBTZUSD'),
        ('kraken_btcusdt', fetch_kraken_ohlcv, 'XBTUSDT'),
        ('kraken_btcusdc', fetch_kraken_ohlcv, 'XBTUSDC'),
        ('kraken_btceur',  fetch_kraken_ohlcv, 'XXBTZEUR'),
        # Kraken Stablecoin Pairs
        ('kraken_usdcusd', fetch_kraken_ohlcv, 'USDCUSD'),
        ('kraken_usdtusd', fetch_kraken_ohlcv, 'USDTZUSD'),
    ]

    for name, fn, symbol in pairs_conf:
        load_or_fetch(name, fn, symbol)
    print("All data fetched.")

if __name__ == '__main__':
    fetch_all_data()

# Uncomment the line below to re-fetch all data:
# fetch_all_data()


Study window: 2023-03-01T00:00:00+00:00 to 2023-03-21T23:59:00+00:00
  Loading cached: binance_btcusdt
  Loading cached: binance_btcusdc
  Loading cached: binance_btceur
  Loading cached: binance_usdcusdt
  Loading cached: coinbase_btcusd
  Loading cached: coinbase_btcusdt
  Loading cached: coinbase_btceur
  Loading cached: coinbase_usdtusd
  Loading cached: kraken_btcusd
  Loading cached: kraken_btcusdt
  Loading cached: kraken_btcusdc
  Loading cached: kraken_btceur
  Loading cached: kraken_usdcusd
  Loading cached: kraken_usdtusd
All data fetched.


---
# Section 2: Data Processing

Builds the aligned 1-minute master dataset from raw exchange data:
- Constructs a common UTC grid (Mar 1–21, 2023, 1-minute frequency)
- Forward-fills prices up to 5 minutes and tracks fill flags
- Computes unadjusted dispersion $D_t = \log(P_{BTC/Q}) - \log(P_{BTC/USD})$
- Computes adjusted parity residual $B_t = \log(P_{BTC/Q} \cdot P_{Q/USD}) - \log(P_{BTC/USD})$
- Computes cross-exchange basis and stablecoin peg deviations

**Source**: `src/02_build_master_data.py`

**Outputs**: 6 parquet files in `data_processed/`


In [3]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timezone

START_DT = datetime(2023, 3, 1, 0, 0, 0, tzinfo=timezone.utc)
END_DT   = datetime(2023, 3, 21, 23, 59, 0, tzinfo=timezone.utc)

DATA_RAW = 'data_raw'
DATA_PROCESSED = 'data_processed'
os.makedirs(DATA_PROCESSED, exist_ok=True)

def load_data(name: str) -> pd.DataFrame:
    path = os.path.join(DATA_RAW, f"{name}.parquet")
    if os.path.exists(path):
        return pd.read_parquet(path)
    return pd.DataFrame()

# 1. Load Data
datasets = {
    'binance_btcusdt': load_data('binance_btcusdt'),
    'binance_btcusdc': load_data('binance_btcusdc'),
    'binance_usdcusdt': load_data('binance_usdcusdt'),

    'coinbase_btcusd': load_data('coinbase_btcusd'),
    'coinbase_btcusdt': load_data('coinbase_btcusdt'),
    'coinbase_usdtusd': load_data('coinbase_usdtusd'),

    'kraken_btcusd': load_data('kraken_btcusd'),
    'kraken_btcusdt': load_data('kraken_btcusdt'),
    'kraken_btcusdc': load_data('kraken_btcusdc'),
    'kraken_usdcusd': load_data('kraken_usdcusd'),
    'kraken_usdtusd': load_data('kraken_usdtusd'),
}

# 2. Build common grid
full_index = pd.date_range(START_DT, END_DT, freq='1min', tz='UTC')

def extract_col(df, col, idx, fill=np.nan):
    if df.empty or col not in df.columns:
        return pd.Series(fill, index=idx, dtype=float)
    return df[col].reindex(idx)

# Forward fill prices up to 5 minutes and track ffill flags
raw_prices = {
    name: extract_col(df, 'close', full_index)
    for name, df in datasets.items()
}

prices = pd.DataFrame(index=full_index)
price_ffill_flags = pd.DataFrame(index=full_index)
for name, raw in raw_prices.items():
    filled = raw.ffill(limit=5)
    prices[name] = filled
    price_ffill_flags[name] = raw.isna() & filled.notna()

# Range Proxy: (High - Low) / Close
ranges = pd.DataFrame({
    name: ((extract_col(df, 'high', full_index) - extract_col(df, 'low', full_index)) / 
           extract_col(df, 'close', full_index)).ffill(limit=5)
    for name, df in datasets.items()
})

volumes = pd.DataFrame({
    name: extract_col(df, 'volume', full_index, fill=0.0)
    for name, df in datasets.items()
})

# 3. Calculate pricing dispersion objects and series-level ffill flags
# Unadjusted dispersion (de-peg-sensitive):
#   d_Q(t) = log(P_BTC/Q) - log(P_BTC/USD)
# Adjusted parity residual (marking stablecoin back to USD):
#   b_Q(t) = log(P_BTC/Q * P_Q/USD) - log(P_BTC/USD)
basis = pd.DataFrame(index=full_index)
basis_ffill_flags = pd.DataFrame(index=full_index)

# ---------------------------------------------------------------------
# KRAKEN: USDC and USDT channels against BTC/USD
# ---------------------------------------------------------------------
# Unadjusted dispersion D_t
basis['dispersion_usdc_kraken'] = (
    np.log(prices['kraken_btcusdc']) - np.log(prices['kraken_btcusd'])
) * 10000
basis_ffill_flags['dispersion_usdc_kraken'] = price_ffill_flags[['kraken_btcusdc', 'kraken_btcusd']].any(axis=1)

basis['dispersion_usdt_kraken'] = (
    np.log(prices['kraken_btcusdt']) - np.log(prices['kraken_btcusd'])
) * 10000
basis_ffill_flags['dispersion_usdt_kraken'] = price_ffill_flags[['kraken_btcusdt', 'kraken_btcusd']].any(axis=1)

# Adjusted parity residual B_t
basis['basis_usdc_kraken'] = (
    np.log(prices['kraken_btcusdc'] * prices['kraken_usdcusd']) - np.log(prices['kraken_btcusd'])
) * 10000 # in bps
basis_ffill_flags['basis_usdc_kraken'] = price_ffill_flags[['kraken_btcusdc', 'kraken_usdcusd', 'kraken_btcusd']].any(axis=1)

basis['basis_usdt_kraken'] = (
    np.log(prices['kraken_btcusdt'] * prices['kraken_usdtusd']) - np.log(prices['kraken_btcusd'])
) * 10000
basis_ffill_flags['basis_usdt_kraken'] = price_ffill_flags[['kraken_btcusdt', 'kraken_usdtusd', 'kraken_btcusd']].any(axis=1)

# ---------------------------------------------------------------------
# COINBASE: USDT channel against BTC/USD
# ---------------------------------------------------------------------
basis['dispersion_usdt_coinbase'] = (
    np.log(prices['coinbase_btcusdt']) - np.log(prices['coinbase_btcusd'])
) * 10000
basis_ffill_flags['dispersion_usdt_coinbase'] = price_ffill_flags[['coinbase_btcusdt', 'coinbase_btcusd']].any(axis=1)

basis['basis_usdt_coinbase'] = (
    np.log(prices['coinbase_btcusdt'] * prices['coinbase_usdtusd']) - np.log(prices['coinbase_btcusd'])
) * 10000
basis_ffill_flags['basis_usdt_coinbase'] = price_ffill_flags[['coinbase_btcusdt', 'coinbase_usdtusd', 'coinbase_btcusd']].any(axis=1)

# BINANCE: It doesn't have fiat USD. We can calculate the relative stablecoin basis (BTC/USDC vs BTC/USDT)
# basis_usdc_vs_usdt_binance = log(P_BTC/USDC * P_USDC/USDT) - log(P_BTC/USDT)
basis['basis_usdc_usdt_binance'] = (
    np.log(prices['binance_btcusdc'] * prices['binance_usdcusdt']) - np.log(prices['binance_btcusdt'])
) * 10000
basis_ffill_flags['basis_usdc_usdt_binance'] = price_ffill_flags[['binance_btcusdc', 'binance_usdcusdt', 'binance_btcusdt']].any(axis=1)

# =====================================================================
# CROSS-EXCHANGE BASIS: Same pair, different exchanges (spatial arb)
# =====================================================================
# Cross-exchange BTC/USDT: Binance vs Kraken
basis['xbasis_btcusdt_binance_kraken'] = (
    np.log(prices['binance_btcusdt']) - np.log(prices['kraken_btcusdt'])
) * 10000
basis_ffill_flags['xbasis_btcusdt_binance_kraken'] = price_ffill_flags[['binance_btcusdt', 'kraken_btcusdt']].any(axis=1)

# Cross-exchange BTC/USDT: Coinbase vs Kraken
basis['xbasis_btcusdt_coinbase_kraken'] = (
    np.log(prices['coinbase_btcusdt']) - np.log(prices['kraken_btcusdt'])
) * 10000
basis_ffill_flags['xbasis_btcusdt_coinbase_kraken'] = price_ffill_flags[['coinbase_btcusdt', 'kraken_btcusdt']].any(axis=1)

# Cross-exchange BTC/USD: Coinbase vs Kraken (fiat vs fiat)
basis['xbasis_btcusd_coinbase_kraken'] = (
    np.log(prices['coinbase_btcusd']) - np.log(prices['kraken_btcusd'])
) * 10000
basis_ffill_flags['xbasis_btcusd_coinbase_kraken'] = price_ffill_flags[['coinbase_btcusd', 'kraken_btcusd']].any(axis=1)

# =====================================================================
# Implied USDT/USD from BTC triangulation (Coinbase)
# =====================================================================
# implied = P_BTC/USD / P_BTC/USDT  (should be ~1.0 if USDT pegged)
prices['implied_usdt_usd_coinbase'] = prices['coinbase_btcusd'] / prices['coinbase_btcusdt']
prices['implied_usdt_usd_kraken']   = prices['kraken_btcusd'] / prices['kraken_btcusdt']
prices['implied_usdc_usd_kraken']   = prices['kraken_btcusd'] / prices['kraken_btcusdc']

# USDT/USD deviation from parity in bps (direct pairs)
basis['usdt_peg_dev_kraken'] = (prices['kraken_usdtusd'] - 1.0) * 10000
basis['usdt_peg_dev_coinbase'] = (prices['coinbase_usdtusd'] - 1.0) * 10000
basis['usdc_peg_dev_kraken'] = (prices['kraken_usdcusd'] - 1.0) * 10000
basis_ffill_flags['usdt_peg_dev_kraken'] = price_ffill_flags['kraken_usdtusd']
basis_ffill_flags['usdt_peg_dev_coinbase'] = price_ffill_flags['coinbase_usdtusd']
basis_ffill_flags['usdc_peg_dev_kraken'] = price_ffill_flags['kraken_usdcusd']

# 4. Save to processed
prices.to_parquet(os.path.join(DATA_PROCESSED, 'prices.parquet'))
price_ffill_flags = price_ffill_flags.reindex(columns=prices.columns).fillna(False).astype(bool)
price_ffill_flags.to_parquet(os.path.join(DATA_PROCESSED, 'price_ffill_flags.parquet'))
ranges.to_parquet(os.path.join(DATA_PROCESSED, 'intraminute_ranges.parquet'))
volumes.to_parquet(os.path.join(DATA_PROCESSED, 'volumes.parquet'))
basis.to_parquet(os.path.join(DATA_PROCESSED, 'basis.parquet'))
basis_ffill_flags = basis_ffill_flags.reindex(columns=basis.columns).fillna(False).astype(bool)
basis_ffill_flags.to_parquet(os.path.join(DATA_PROCESSED, 'basis_ffill_flags.parquet'))

print("Master datasets built and saved to data_processed/")

print("Checking for degenerate basis outputs:")
for col in basis.columns:
    valid = basis[col].dropna()
    print(f"{col}: {len(valid)} non-NaN, Mean: {valid.mean():.4f}, Std: {valid.std():.4f}, Unique vals: {valid.nunique()}")
    if valid.nunique() < 10:
        print(f"  --> WARNING: degenerate or broken basis detected for {col}")


Master datasets built and saved to data_processed/
Checking for degenerate basis outputs:
dispersion_usdc_kraken: 28441 non-NaN, Mean: 53.4864, Std: 169.5105, Unique vals: 26876
dispersion_usdt_kraken: 29768 non-NaN, Mean: -21.5626, Std: 27.9859, Unique vals: 28285
basis_usdc_kraken: 27900 non-NaN, Mean: 0.8811, Std: 10.9167, Unique vals: 26843
basis_usdt_kraken: 29768 non-NaN, Mean: -0.6831, Std: 5.7140, Unique vals: 28890
dispersion_usdt_coinbase: 29915 non-NaN, Mean: -23.7378, Std: 29.2394, Unique vals: 29850
basis_usdt_coinbase: 29904 non-NaN, Mean: -0.0463, Std: 4.4658, Unique vals: 29880
basis_usdc_usdt_binance: 30240 non-NaN, Mean: -2.9498, Std: 25.4530, Unique vals: 29621
xbasis_btcusdt_binance_kraken: 29768 non-NaN, Mean: 0.0069, Std: 5.9258, Unique vals: 29369
xbasis_btcusdt_coinbase_kraken: 29453 non-NaN, Mean: 0.0100, Std: 6.4839, Unique vals: 26979
xbasis_btcusd_coinbase_kraken: 29961 non-NaN, Mean: 2.1846, Std: 6.2044, Unique vals: 29906
usdt_peg_dev_kraken: 30233 non-NaN

---
# Section 3: Half-Life Utility Functions

Shared utility functions for OU/AR(1) half-life estimation used throughout the analysis.

**Source**: `src/half_life_utils.py`


In [4]:
import numpy as np
import pandas as pd
import statsmodels.api as sm


def half_life_from_rho(rho: float, dt_minutes: float) -> float:
    """
    Exact half-life mapping for a levels AR(1):
      X_t = c + rho * X_{t-1} + eps_t

    half_life = (ln(2) * dt_minutes) / (-ln(rho))
    Valid mean-reverting domain: 0 < rho < 1.
    """
    if dt_minutes <= 0:
        raise ValueError(f"dt_minutes must be positive; got {dt_minutes}")
    if not np.isfinite(rho) or rho <= 0.0 or rho >= 1.0:
        return np.nan
    return float((np.log(2.0) * dt_minutes) / (-np.log(rho)))


def estimate_half_life_from_ecm(
    series: pd.Series,
    dt_minutes: float,
    ff_mask: pd.Series | None = None,
    min_obs: int = 100,
) -> dict:
    """
    Estimate half-life via ECM regression:
      ΔX_t = a + beta * X_{t-1} + eps_t
    which implies levels AR(1):
      X_t = c + rho * X_{t-1} + eps_t,  rho = 1 + beta

    If ff_mask is provided, any observation where X_t or X_{t-1}
    is forward-filled is removed from the regression sample.
    """
    s = series.dropna().astype(float)
    if len(s) < min_obs:
        return {
            'estimation_form': 'ECM (implied AR1)',
            'beta_est': np.nan,
            'kappa_est': np.nan,
            'rho_est': np.nan,
            'half_life_min': np.nan,
            'n_obs': len(s),
            'warning': 'obs_too_few',
        }

    reg = pd.DataFrame({'x_t': s})
    reg['x_lag'] = reg['x_t'].shift(1)
    reg['dx_t'] = reg['x_t'] - reg['x_lag']

    if ff_mask is not None:
        ff = ff_mask.reindex(reg.index).fillna(False).astype(bool)
        reg['ff_t'] = ff
        reg['ff_lag'] = ff.shift(1, fill_value=False).astype(bool)
    else:
        reg['ff_t'] = False
        reg['ff_lag'] = False

    reg = reg.dropna(subset=['dx_t', 'x_lag'])
    reg = reg[~(reg['ff_t'] | reg['ff_lag'])]

    if len(reg) < min_obs:
        return {
            'estimation_form': 'ECM (implied AR1)',
            'beta_est': np.nan,
            'kappa_est': np.nan,
            'rho_est': np.nan,
            'half_life_min': np.nan,
            'n_obs': len(reg),
            'warning': 'obs_too_few_no_ff',
        }

    X = sm.add_constant(reg['x_lag'])
    model = sm.OLS(reg['dx_t'], X).fit()
    beta = float(model.params.iloc[1])
    kappa = float(-beta)
    rho = float(1.0 + beta)
    hl = half_life_from_rho(rho, dt_minutes=dt_minutes)

    warning = ''
    if not (0.0 < rho < 1.0):
        warning = 'rho_invalid'

    return {
        'estimation_form': 'ECM (implied AR1)',
        'beta_est': beta,
        'kappa_est': kappa,
        'rho_est': rho,
        'half_life_min': hl,
        'n_obs': int(len(reg)),
        'warning': warning,
    }


def run_half_life_sanity_tests(dt_minutes: float = 1.0) -> pd.DataFrame:
    """
    Unit-style checks:
    - monotonicity in rho grid
    - undefined outside (0,1)
    """
    rho_grid = [0.5, 0.8, 0.9, 0.95, 0.99]
    hl_grid = [half_life_from_rho(rho, dt_minutes=dt_minutes) for rho in rho_grid]

    if not all(np.isfinite(v) for v in hl_grid):
        raise AssertionError(f"Expected finite half-lives on rho grid; got {hl_grid}")
    if not all(hl_grid[i] < hl_grid[i + 1] for i in range(len(hl_grid) - 1)):
        raise AssertionError(f"Half-life is not strictly increasing in rho: {hl_grid}")
    if hl_grid[-1] <= 10.0 * hl_grid[0]:
        raise AssertionError(
            "Half-life does not expand enough near rho -> 1; expected blow-up behavior."
        )

    invalid_rhos = [-0.2, 0.0, 1.0, 1.1]
    invalid_hl = [half_life_from_rho(rho, dt_minutes=dt_minutes) for rho in invalid_rhos]
    if not all(np.isnan(v) for v in invalid_hl):
        raise AssertionError(f"Invalid rhos should return NaN half-life; got {invalid_hl}")

    out = pd.DataFrame({
        'rho': rho_grid,
        'dt_minutes': dt_minutes,
        'half_life_min': hl_grid,
    })
    return out


---
# Section 4: Load Processed Data & Define Regimes

Load the processed datasets and define the three-regime structure used throughout.


In [5]:
# Load processed data
prices = pd.read_parquet(os.path.join(DATA_PROCESSED, 'prices.parquet'))

price_ff_flags_path = os.path.join(DATA_PROCESSED, 'price_ffill_flags.parquet')
if os.path.exists(price_ff_flags_path):
    price_ff_flags = pd.read_parquet(price_ff_flags_path)
else:
    price_ff_flags = pd.DataFrame(False, index=prices.index, columns=prices.columns)
price_ff_flags = price_ff_flags.reindex(index=prices.index, columns=prices.columns).fillna(False).astype(bool)

ranges = pd.read_parquet(os.path.join(DATA_PROCESSED, 'intraminute_ranges.parquet'))
volumes = pd.read_parquet(os.path.join(DATA_PROCESSED, 'volumes.parquet'))
basis = pd.read_parquet(os.path.join(DATA_PROCESSED, 'basis.parquet'))

basis_ff_flags_path = os.path.join(DATA_PROCESSED, 'basis_ffill_flags.parquet')
if os.path.exists(basis_ff_flags_path):
    basis_ff_flags = pd.read_parquet(basis_ff_flags_path)
else:
    basis_ff_flags = pd.DataFrame(False, index=basis.index, columns=basis.columns)
basis_ff_flags = basis_ff_flags.reindex(index=basis.index, columns=basis.columns).fillna(False).astype(bool)

returns = prices.pct_change(fill_method=None).dropna()

# Regime boundaries
svb_start = pd.Timestamp('2023-03-10', tz='UTC')
svb_end   = pd.Timestamp('2023-03-13', tz='UTC')
regimes = {
    'Pre-SVB':  (prices.index.min(), svb_start),
    'Crisis':   (svb_start, svb_end),
    'Post-SVB': (svb_end, prices.index.max()),
}

def assign_regime(idx):
    if idx < svb_start: return 'Pre-SVB'
    elif idx < svb_end: return 'Crisis'
    else: return 'Post-SVB'

REGIME_ORDER  = ['Pre-SVB', 'Crisis', 'Post-SVB']
REGIME_COLORS = {'Pre-SVB': '#3498db', 'Crisis': '#e74c3c', 'Post-SVB': '#2ecc71'}

# Regime masks (used throughout)
m_all  = prices.index == prices.index
m_pre  = prices.index < svb_start
m_cri  = (prices.index >= svb_start) & (prices.index < svb_end)
m_post = prices.index >= svb_end
m_calm = m_pre | m_post

print(f'Prices shape: {prices.shape}')
print(f'Basis shape:  {basis.shape}')
print(f'Date range:   {prices.index.min()} to {prices.index.max()}')


Prices shape: (30240, 14)
Basis shape:  (30240, 13)
Date range:   2023-03-01 00:00:00+00:00 to 2023-03-21 23:59:00+00:00


---
# Section 5: Core Analysis and Tables

This section contains the core statistical analysis from `src/03_analysis_and_figures.py`:
- Identity check between $D_t$ and $B_t$
- Regime statistics for dispersion and adjusted residuals (Table 2 in paper)
- OU/AR(1) mean-reversion estimation and ADF tests
- Half-life robustness (1m vs 5m, all vs no-forward-fill) (Figure 4 in paper)
- HAC regressions of $B_t$ on crisis dummy, volatility, and range proxy (Table 6 in paper)
- Johansen cointegration and VECM price discovery (Table 7 in paper)
- Granger causality tests with BH/FDR correction
- Arbitrage profitability analysis (Table 8, Figure 14 in paper)


### 5.1 Helper Functions & Identity Check


In [6]:
# Helper functions from 03_analysis_and_figures.py

def build_regime_stats(df, series_map, regimes_dict):
    out = []
    for regime, (t0, t1) in regimes_dict.items():
        mask = (df.index >= t0) & (df.index < t1)
        for col, lbl in series_map:
            if col not in df.columns:
                continue
            s = df.loc[mask, col].dropna()
            if len(s) == 0:
                continue
            out.append({
                'Regime': regime,
                'Series': lbl,
                'Mean (bps)': round(s.mean(), 2),
                'Std (bps)': round(s.std(), 2),
                'Mean Abs (bps)': round(s.abs().mean(), 2),
                'N': len(s),
            })
    return pd.DataFrame(out)


def make_width_safe_latex(latex_text, add_footnotesize=False):
    if add_footnotesize:
        latex_text = latex_text.replace('\\begin{tabular}', '\\footnotesize\n\\begin{tabular}', 1)
    latex_text = latex_text.replace(
        '\\begin{tabular}',
        '\\resizebox{\\textwidth}{!}{%\n\\begin{tabular}', 1)
    latex_text = latex_text.replace('\\end{tabular}', '\\end{tabular}%\n}', 1)
    return latex_text


def gg_component_share_from_alpha(alpha_vec):
    if len(alpha_vec) != 2:
        raise ValueError('Gonzalo-Granger helper expects exactly 2 alpha coefficients.')
    a1 = float(alpha_vec[0])
    a2 = float(alpha_vec[1])
    denom = a2 - a1
    if np.isclose(denom, 0.0):
        return np.nan, np.nan, 'gg_denominator_near_zero'
    s1 = a2 / denom
    s2 = -a1 / denom
    warning = ''
    if (not np.isfinite(s1)) or (not np.isfinite(s2)):
        warning = 'gg_non_finite'
    elif (s1 < 0.0) or (s1 > 1.0) or (s2 < 0.0) or (s2 > 1.0):
        warning = 'gg_non_convex_share'
    return float(s1), float(s2), warning


# Half-life sanity check
df_hl_sanity = run_half_life_sanity_tests(dt_minutes=1.0)
df_hl_sanity.to_csv(os.path.join(TABLES_DIR, 'half_life_sanity_grid.csv'), index=False)
print('Half-life sanity tests passed.')
print(df_hl_sanity)


Half-life sanity tests passed.
    rho  dt_minutes  half_life_min
0  0.50         1.0       1.000000
1  0.80         1.0       3.106284
2  0.90         1.0       6.578813
3  0.95         1.0      13.513407
4  0.99         1.0      68.967564


In [7]:
# Identity check: B_t - D_t - log(P_stablecoin/USD) * 10000 should be ~0
identity_specs = [
    ('USDC (Kraken)', 'basis_usdc_kraken', 'dispersion_usdc_kraken', 'kraken_usdcusd'),
    ('USDT (Kraken)', 'basis_usdt_kraken', 'dispersion_usdt_kraken', 'kraken_usdtusd'),
    ('USDT (Coinbase)', 'basis_usdt_coinbase', 'dispersion_usdt_coinbase', 'coinbase_usdtusd'),
]

identity_rows = []
for market, b_col, d_col, peg_col in identity_specs:
    if not ({b_col, d_col}.issubset(basis.columns) and peg_col in prices.columns):
        continue
    aligned = pd.concat(
        [basis[b_col], basis[d_col], prices[peg_col]],
        axis=1, keys=['B', 'D', 'peg']
    ).dropna()
    if aligned.empty:
        continue
    residual = aligned['B'] - aligned['D'] - np.log(aligned['peg']) * 10000
    identity_rows.append({
        'Market': market,
        'N': len(residual),
        'Mean Identity Error (bps)': float(residual.mean()),
        'Max Abs Identity Error (bps)': float(residual.abs().max()),
        'Std Identity Error (bps)': float(residual.std()),
    })

df_identity = pd.DataFrame(identity_rows)
df_identity.to_csv(os.path.join(TABLES_DIR, 'dispersion_adjusted_identity_check.csv'), index=False)

IDENTITY_TOL_BPS = 1e-6
assert not (df_identity['Max Abs Identity Error (bps)'] > IDENTITY_TOL_BPS).any(), \
    f'Identity check failed!\n{df_identity.to_string(index=False)}'

print('Identity check passed:')
print(df_identity.to_string(index=False))


Identity check passed:
         Market     N  Mean Identity Error (bps)  Max Abs Identity Error (bps)  Std Identity Error (bps)
  USDC (Kraken) 27900               8.416706e-14                  1.831824e-11              6.510114e-12
  USDT (Kraken) 29768              -4.120173e-14                  1.787359e-11              6.456026e-12
USDT (Coinbase) 29904              -2.729364e-14                  1.791278e-11              6.963300e-12


### 5.2 Regime Statistics — Dispersion $D_t$ vs Adjusted Residual $B_t$ (Table 2)


In [8]:
# Table: D_t vs B_t regime statistics (Kraken) — used in Table 2 of column paper
basis['Regime'] = basis.index.map(assign_regime)

series_map_dispersion_adjusted = [
    ('dispersion_usdc_kraken', 'USDC Kraken $D_t$ (Unadjusted)'),
    ('basis_usdc_kraken', 'USDC Kraken $B_t$ (Adjusted)'),
    ('dispersion_usdt_kraken', 'USDT Kraken $D_t$ (Unadjusted)'),
    ('basis_usdt_kraken', 'USDT Kraken $B_t$ (Adjusted)'),
]
df_disp_adj = build_regime_stats(basis, series_map_dispersion_adjusted, regimes)
df_disp_adj.to_csv(os.path.join(TABLES_DIR, 'dispersion_adjusted_stats.csv'), index=False)
with open(os.path.join(TABLES_DIR, 'dispersion_adjusted_stats.tex'), 'w') as f:
    f.write(df_disp_adj.to_latex(
        index=False,
        caption='Regime Statistics for Unadjusted Dispersion ($D_t$) and Adjusted Residual ($B_t$), Kraken',
        label='tab:dispersion_vs_adjusted',
        column_format='llrrrr',
        float_format='%.2f'
    ))

print('Table: Dispersion vs Adjusted Residual Stats')
print(df_disp_adj.to_string(index=False))


Table: Dispersion vs Adjusted Residual Stats
  Regime                         Series  Mean (bps)  Std (bps)  Mean Abs (bps)     N
 Pre-SVB USDC Kraken $D_t$ (Unadjusted)        0.12       4.95            3.49 11615
 Pre-SVB   USDC Kraken $B_t$ (Adjusted)       -0.32       5.00            3.53 11197
 Pre-SVB USDT Kraken $D_t$ (Unadjusted)       -0.54       4.49            3.17 12618
 Pre-SVB   USDT Kraken $B_t$ (Adjusted)       -0.60       4.33            3.02 12618
  Crisis USDC Kraken $D_t$ (Unadjusted)      319.97     322.87          321.17  4311
  Crisis   USDC Kraken $B_t$ (Adjusted)        5.27      21.21           13.37  4283
  Crisis USDT Kraken $D_t$ (Unadjusted)      -57.18      45.10           58.57  4312
  Crisis   USDT Kraken $B_t$ (Adjusted)        0.68       6.57            4.94  4312
Post-SVB USDC Kraken $D_t$ (Unadjusted)       11.22      21.95           14.27 12514
Post-SVB   USDC Kraken $B_t$ (Adjusted)        0.45       9.06            6.68 12419
Post-SVB USDT Kraken

### 5.3 OU/AR(1) Mean-Reversion & Half-Life Robustness (Figure 4)


In [9]:
# OU/AR(1) stats by regime
all_basis_cols = [c for c in basis.columns if c != 'Regime']
stats_list = []
for regime, (t0, t1) in regimes.items():
    mask = (basis.index >= t0) & (basis.index < t1)
    for col in all_basis_cols:
        series = basis.loc[mask, col]
        clean = series.dropna()
        if len(clean) < 100:
            continue
        est = estimate_half_life_from_ecm(series=series, dt_minutes=1.0, ff_mask=None, min_obs=100)
        adf_stat, adf_p = adfuller(clean, maxlag=5)[:2]
        stats_list.append({
            'Regime': regime, 'Basis': col,
            'Mean (bps)': round(clean.mean(), 2), 'Std (bps)': round(clean.std(), 2),
            'rho_est': round(est['rho_est'], 6) if np.isfinite(est['rho_est']) else np.nan,
            'Half-Life (min)': round(est['half_life_min'], 2) if np.isfinite(est['half_life_min']) else np.nan,
            'ADF Stat': round(adf_stat, 2), 'ADF p-value': f'{adf_p:.4f}',
            'N': len(clean), 'Warning': est['warning'],
        })

df_ou = pd.DataFrame(stats_list)
df_ou.to_csv(os.path.join(TABLES_DIR, 'ou_basis_stats.csv'), index=False)
print('OU/AR(1) stats computed and saved.')
print(df_ou[['Regime','Basis','Mean (bps)','Std (bps)','Half-Life (min)','ADF Stat']].to_string(index=False))


OU/AR(1) stats computed and saved.
  Regime                          Basis  Mean (bps)  Std (bps)  Half-Life (min)  ADF Stat
 Pre-SVB         dispersion_usdc_kraken        0.12       4.95             1.01    -43.80
 Pre-SVB         dispersion_usdt_kraken       -0.54       4.49             1.00    -34.12
 Pre-SVB              basis_usdc_kraken       -0.32       5.00             0.98    -40.28
 Pre-SVB              basis_usdt_kraken       -0.60       4.33             0.88    -43.26
 Pre-SVB       dispersion_usdt_coinbase       -0.28       2.95             0.80    -29.61
 Pre-SVB            basis_usdt_coinbase       -0.07       2.67             0.57    -39.96
 Pre-SVB        basis_usdc_usdt_binance       -4.14      25.02             4.92    -26.74
 Pre-SVB  xbasis_btcusdt_binance_kraken       -0.08       4.56             0.80    -44.87
 Pre-SVB xbasis_btcusdt_coinbase_kraken       -0.08       4.98             0.86    -41.81
 Pre-SVB  xbasis_btcusd_coinbase_kraken       -0.33       2.42   

In [10]:
# Half-life robustness: 1m vs 5m, all vs no-forward-fill
robust_series = [
    ('basis_usdc_kraken', 'USDC/USD $B_t$ (Kraken)'),
    ('basis_usdt_kraken', 'USDT/USD $B_t$ (Kraken)'),
]
robust_rows = []

for regime, (t0, t1) in regimes.items():
    regime_mask = (basis.index >= t0) & (basis.index < t1)
    for col, series_label in robust_series:
        if col not in basis.columns:
            continue
        s_1m = basis.loc[regime_mask, col]
        ff_1m = basis_ff_flags.loc[regime_mask, col] if col in basis_ff_flags.columns else pd.Series(False, index=s_1m.index)
        freq_configs = [
            ('1m', 1.0, s_1m, ff_1m),
            ('5m', 5.0, s_1m.resample('5min').last(),
             ff_1m.astype(float).resample('5min').last().fillna(0.0).astype(bool)),
        ]
        for freq_label, dt_minutes, s_freq, ff_freq in freq_configs:
            for ff_filter, ff_arg in [('all', None), ('no_ff', ff_freq)]:
                est = estimate_half_life_from_ecm(series=s_freq, dt_minutes=dt_minutes,
                                                  ff_mask=ff_arg, min_obs=80)
                robust_rows.append({
                    'series': series_label, 'regime': regime, 'freq': freq_label,
                    'ff_filter': ff_filter, 'rho_est': est['rho_est'],
                    'half_life_min': est['half_life_min'], 'n_obs': est['n_obs'],
                    'warning': est['warning'],
                })

df_hl_robust = pd.DataFrame(robust_rows)
df_hl_robust.to_csv(os.path.join(TABLES_DIR, 'half_life_robustness.csv'), index=False)
print('Half-life robustness table saved.')
print(df_hl_robust.to_string(index=False))


Half-life robustness table saved.
                 series   regime freq ff_filter   rho_est  half_life_min  n_obs     warning
USDC/USD $B_t$ (Kraken)  Pre-SVB   1m       all  0.494420       0.984067  11196            
USDC/USD $B_t$ (Kraken)  Pre-SVB   1m     no_ff  0.139676       0.352132   1166            
USDC/USD $B_t$ (Kraken)  Pre-SVB   5m       all  0.072240       1.318894   2478            
USDC/USD $B_t$ (Kraken)  Pre-SVB   5m     no_ff -0.131116            NaN    208 rho_invalid
USDT/USD $B_t$ (Kraken)  Pre-SVB   1m       all  0.454311       0.878544  12617            
USDT/USD $B_t$ (Kraken)  Pre-SVB   1m     no_ff  0.235961       0.479989   4619            
USDT/USD $B_t$ (Kraken)  Pre-SVB   5m       all  0.099781       1.503721   2583            
USDT/USD $B_t$ (Kraken)  Pre-SVB   5m     no_ff  0.143256       1.783590    863            
USDC/USD $B_t$ (Kraken)   Crisis   1m       all  0.320260       0.608760   4282            
USDC/USD $B_t$ (Kraken)   Crisis   1m     no_f

### 5.4 HAC Regressions (Table 6)


In [11]:
# HAC regression: B_t on Crisis dummy, RealizedVol, RangeProxy
# USDC channel
df_reg = pd.DataFrame()
df_reg['Basis'] = basis['basis_usdc_kraken']
df_reg['Crisis'] = (basis['Regime'] == 'Crisis').astype(int)
df_reg['RealizedVol60m'] = returns['kraken_btcusdc'].rolling(60).std() * 10000
df_reg['RangeProxy'] = ranges['kraken_btcusdc'] * 10000
df_reg = df_reg.dropna()

X = sm.add_constant(df_reg[['Crisis', 'RealizedVol60m', 'RangeProxy']])
y = df_reg['Basis']
model_usdc = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 60})

# USDT channel
df_reg2 = pd.DataFrame()
df_reg2['Basis'] = basis['basis_usdt_kraken']
df_reg2['Crisis'] = (basis['Regime'] == 'Crisis').astype(int)
df_reg2['RealizedVol60m'] = returns['kraken_btcusdt'].rolling(60).std() * 10000
df_reg2['RangeProxy'] = ranges['kraken_btcusdt'] * 10000
df_reg2 = df_reg2.dropna()

X2 = sm.add_constant(df_reg2[['Crisis', 'RealizedVol60m', 'RangeProxy']])
y2 = df_reg2['Basis']
model_usdt = sm.OLS(y2, X2).fit(cov_type='HAC', cov_kwds={'maxlags': 60})

# Generate LaTeX table
def fmt_coef(x): return f'${x:+.3f}$'
def fmt_pval(x): return '$<0.001$' if x < 0.001 else f'{x:.3f}'

reg_rows_spec = [('const', 'Constant'), ('Crisis', 'Crisis'),
                 ('RealizedVol60m', 'RealizedVol (60m)'), ('RangeProxy', 'Range Proxy')]
reg_lines = [
    r'\begin{table}[H]',
    r'\caption{HAC Regressions of Adjusted Residual $B_t$ on Crisis Dummy, Realized Volatility, and Range Proxy (Kraken, Newey--West 60 lags)}',
    r'\label{tab:regression_hac}', r'\footnotesize', r'\centering',
    r'\begin{tabular}{lcccc}', r'\toprule',
    r' & \multicolumn{2}{c}{USDC Channel} & \multicolumn{2}{c}{USDT Channel} \\',
    r'\cmidrule(lr){2-3} \cmidrule(lr){4-5}',
    r' & Coef. & $p$-value & Coef. & $p$-value \\', r'\midrule',
]
for key, label in reg_rows_spec:
    reg_lines.append(
        f'{label:<17} & {fmt_coef(float(model_usdc.params[key]))} & {fmt_pval(float(model_usdc.pvalues[key]))} & '
        f'{fmt_coef(float(model_usdt.params[key]))} & {fmt_pval(float(model_usdt.pvalues[key]))} \\\\'
    )
reg_lines.extend([
    r'\midrule',
    f'$R^2$             & \\multicolumn{{2}}{{c}}{{{model_usdc.rsquared:.3f}}} & \\multicolumn{{2}}{{c}}{{{model_usdt.rsquared:.3f}}} \\\\',
    f'$N$               & \\multicolumn{{2}}{{c}}{{{int(model_usdc.nobs):,}}} & \\multicolumn{{2}}{{c}}{{{int(model_usdt.nobs):,}}} \\\\',
    r'\bottomrule',
    r"\multicolumn{5}{l}{\footnotesize OLS with HAC standard errors (Newey--West, 60 lags). Dependent variable: $B_t$ (bps).}",
    r'\end{tabular}', r'\end{table}',
])
with open(os.path.join(TABLES_DIR, 'regression_hac.tex'), 'w') as f:
    f.write('\n'.join(reg_lines) + '\n')

print('USDC Channel HAC Regression:')
print(model_usdc.summary().tables[1])
print('\nUSDT Channel HAC Regression:')
print(model_usdt.summary().tables[1])


USDC Channel HAC Regression:
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -0.0788      0.374     -0.211      0.833      -0.812       0.655
Crisis             4.2954      0.819      5.248      0.000       2.691       5.900
RealizedVol60m     0.0044      0.048      0.091      0.928      -0.090       0.099
RangeProxy         0.0480      0.017      2.810      0.005       0.015       0.082

USDT Channel HAC Regression:
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -0.6562      0.167     -3.940      0.000      -0.983      -0.330
Crisis             1.6547      0.380      4.360      0.000       0.911       2.399
RealizedVol60m    -0.0294      0.022     -1.345      0.179      -0.072       0.013
RangeProxy        -0.0125   

### 5.5 Johansen Cointegration & VECM Price Discovery (Table 7)


In [12]:
# Johansen cointegration and VECM price discovery
primary_channels = [
    {'channel': 'Kraken BTC/USD vs BTC/USDC', 'market_1': 'Kraken BTC/USD',
     'market_2': 'Kraken BTC/USDC', 'col_1': 'kraken_btcusd', 'col_2': 'kraken_btcusdc'},
    {'channel': 'Kraken BTC/USD vs BTC/USDT', 'market_1': 'Kraken BTC/USD',
     'market_2': 'Kraken BTC/USDT', 'col_1': 'kraken_btcusd', 'col_2': 'kraken_btcusdt'},
]

johansen_rows = []
discovery_rows = []

def johansen_rank_summary(endog_levels, k_ar_diff):
    joh = coint_johansen(endog_levels, det_order=0, k_ar_diff=k_ar_diff)
    trace_r0 = float(joh.lr1[0])
    crit95_r0 = float(joh.cvt[0, 1])
    trace_r1 = float(joh.lr1[1])
    crit95_r1 = float(joh.cvt[1, 1])
    reject_r0 = trace_r0 > crit95_r0
    reject_r1 = trace_r1 > crit95_r1
    rank_95 = int(reject_r0) + int(reject_r1)
    return {'trace_stat_r0': trace_r0, 'trace_stat_r1': trace_r1,
            'trace_crit95_r0': crit95_r0, 'trace_crit95_r1': crit95_r1,
            'reject_r0_95': reject_r0, 'reject_r1_95': reject_r1, 'rank_95': rank_95}

for ch in primary_channels:
    c1, c2 = ch['col_1'], ch['col_2']
    if c1 not in prices.columns or c2 not in prices.columns:
        continue
    df_levels = pd.DataFrame({'p1': np.log(prices[c1]), 'p2': np.log(prices[c2])}, index=prices.index)
    ff_mask = pd.Series(False, index=prices.index)
    if c1 in price_ff_flags.columns: ff_mask = ff_mask | price_ff_flags[c1]
    if c2 in price_ff_flags.columns: ff_mask = ff_mask | price_ff_flags[c2]
    df_levels = df_levels[~ff_mask].dropna()
    endog_levels = df_levels[['p1', 'p2']].to_numpy()

    sel = select_order(endog_levels, maxlags=10, deterministic='ci')
    p_aic = sel.selected_orders.get('aic')
    p_bic = sel.selected_orders.get('bic')
    p_used = p_bic if p_bic is not None else (p_aic if p_aic is not None else 2)
    p_used = int(max(p_used, 1))
    k_ar_diff = max(p_used - 1, 0)

    joh_base = johansen_rank_summary(endog_levels, k_ar_diff=k_ar_diff)
    rank_95 = joh_base['rank_95']
    rank_used = min(rank_95, 1)

    johansen_rows.append({
        'channel': ch['channel'], 'n_obs_no_ff': int(len(df_levels)),
        'selected_p_used': p_used, 'k_ar_diff_used': k_ar_diff,
        'trace_stat_r0': joh_base['trace_stat_r0'],
        'trace_crit95_r0': joh_base['trace_crit95_r0'],
        'reject_r0_95': joh_base['reject_r0_95'], 'rank_used': rank_used,
    })

    if rank_used < 1:
        discovery_rows.append({'channel': ch['channel'], 'rank_used': rank_used,
                               'alpha_market_1': np.nan, 'alpha_market_2': np.nan,
                               'leader_by_adjustment': 'undetermined_no_cointegration'})
        continue

    vecm = VECM(endog_levels, k_ar_diff=k_ar_diff, coint_rank=rank_used, deterministic='ci').fit()
    alpha = vecm.alpha[:, 0]
    a1, a2 = float(alpha[0]), float(alpha[1])
    if np.isclose(abs(a1), abs(a2)): leader = 'co-adjusting'
    elif abs(a1) < abs(a2): leader = ch['market_1']
    else: leader = ch['market_2']
    gg1, gg2, gg_warning = gg_component_share_from_alpha(alpha)

    discovery_rows.append({
        'channel': ch['channel'], 'rank_used': rank_used,
        'alpha_market_1': a1, 'alpha_market_2': a2,
        'gg_share_market_1': gg1, 'gg_share_market_2': gg2,
        'leader_by_adjustment': leader, 'gg_warning': gg_warning,
    })
    print(f'{ch["channel"]}: rank={rank_used}, alpha=[{a1:.5f}, {a2:.5f}], leader={leader}')

df_johansen = pd.DataFrame(johansen_rows)
df_johansen.to_csv(os.path.join(TABLES_DIR, 'cointegration_johansen.csv'), index=False)
df_discovery = pd.DataFrame(discovery_rows)
df_discovery.to_csv(os.path.join(TABLES_DIR, 'price_discovery_metrics.csv'), index=False)
print('\nJohansen cointegration results:')
print(df_johansen.to_string(index=False))


Kraken BTC/USD vs BTC/USDT: rank=1, alpha=[0.00786, 0.01182], leader=Kraken BTC/USD

Johansen cointegration results:
                   channel  n_obs_no_ff  selected_p_used  k_ar_diff_used  trace_stat_r0  trace_crit95_r0  reject_r0_95  rank_used
Kraken BTC/USD vs BTC/USDC        15710                3               2       7.705162          15.4943         False          0
Kraken BTC/USD vs BTC/USDT        20415                9               8      27.025736          15.4943          True          1


### 5.6 Granger Causality Tests


In [13]:
# Granger causality tests
granger_pairs = [
    ('kraken_btcusd',    'kraken_btcusdc',  'BTC/USDC → BTC/USD (Kraken)'),
    ('kraken_btcusdc',   'kraken_btcusd',   'BTC/USD → BTC/USDC (Kraken)'),
    ('kraken_btcusd',    'kraken_btcusdt',  'BTC/USDT → BTC/USD (Kraken)'),
    ('kraken_btcusdt',   'kraken_btcusd',   'BTC/USD → BTC/USDT (Kraken)'),
    ('binance_btcusdt',  'kraken_btcusdt',  'Kraken USDT → Binance USDT'),
    ('kraken_btcusdt',   'binance_btcusdt', 'Binance USDT → Kraken USDT'),
    ('coinbase_btcusd',  'kraken_btcusd',   'Kraken USD → Coinbase USD'),
    ('kraken_btcusd',    'coinbase_btcusd', 'Coinbase USD → Kraken USD'),
]

granger_results = []
for dep, indep, label in granger_pairs:
    if dep in returns.columns and indep in returns.columns:
        var_data = returns[[dep, indep]].dropna() * 10000
        if len(var_data) < 200: continue
        try:
            var_model = VAR(var_data)
            res = var_model.fit(maxlags=10, ic='aic')
            g_test = res.test_causality(dep, indep, kind='f')
            granger_results.append({
                'Test': label, 'VAR Lags': res.k_ar,
                'F-stat': round(g_test.test_statistic, 3),
                'p-value': float(g_test.pvalue),
                'Significant': '***' if g_test.pvalue < 0.001 else ('**' if g_test.pvalue < 0.01 else ('*' if g_test.pvalue < 0.05 else '')),
            })
        except Exception as e:
            print(f'Granger test failed for {label}: {e}')

df_granger = pd.DataFrame(granger_results)
df_granger.to_csv(os.path.join(TABLES_DIR, 'granger_causality.csv'), index=False)

# BH/FDR correction
if not df_granger.empty:
    _, qvals, _, _ = multipletests(df_granger['p-value'].values, method='fdr_bh')
    df_granger_fdr = df_granger.copy()
    df_granger_fdr['q-value (BH/FDR)'] = qvals
    df_granger_fdr['Significant FDR'] = df_granger_fdr['q-value (BH/FDR)'].apply(lambda q: 'Yes' if q < 0.05 else 'No')
    df_granger_fdr.to_csv(os.path.join(TABLES_DIR, 'granger_causality_fdr.csv'), index=False)
    print(df_granger_fdr.to_string(index=False))


                       Test  VAR Lags  F-stat       p-value Significant  q-value (BH/FDR) Significant FDR
BTC/USDC → BTC/USD (Kraken)        10   1.512  1.278975e-01                  1.278975e-01              No
BTC/USD → BTC/USDC (Kraken)        10 332.178  0.000000e+00         ***      0.000000e+00             Yes
BTC/USDT → BTC/USD (Kraken)        10   3.825  3.446514e-05         ***      4.595352e-05             Yes
BTC/USD → BTC/USDT (Kraken)        10 385.271  0.000000e+00         ***      0.000000e+00             Yes
 Kraken USDT → Binance USDT        10   7.265  1.389495e-11         ***      2.223192e-11             Yes
 Binance USDT → Kraken USDT        10 476.065  0.000000e+00         ***      0.000000e+00             Yes
  Kraken USD → Coinbase USD        10   3.709  5.477398e-05         ***      6.259883e-05             Yes
  Coinbase USD → Kraken USD        10 130.133 4.617437e-270         ***     9.234875e-270             Yes


### 5.7 Arbitrage Profitability Analysis (Table 8)


In [14]:
# Arbitrage after fees
FEE_BPS_PER_LEG = 5.0
arb_channel_specs = [
    {'channel_key': 'basis_usdc_kraken', 'basis_col': 'basis_usdc_kraken',
     'label_short': 'USDC/USD Kraken (3-leg)', 'label_table': 'USDC/USD (Kraken, 3-leg triangular)',
     'n_legs': 3, 'range_leg_cols': ['kraken_btcusdc', 'kraken_usdcusd', 'kraken_btcusd']},
    {'channel_key': 'basis_usdt_kraken', 'basis_col': 'basis_usdt_kraken',
     'label_short': 'USDT/USD Kraken (3-leg)', 'label_table': 'USDT/USD (Kraken, 3-leg triangular)',
     'n_legs': 3, 'range_leg_cols': ['kraken_btcusdt', 'kraken_usdtusd', 'kraken_btcusd']},
    {'channel_key': 'xbasis_btcusdt_binance_kraken', 'basis_col': 'xbasis_btcusdt_binance_kraken',
     'label_short': 'Cross BTC/USDT Bin-Kra (2-leg)', 'label_table': 'Cross BTC/USDT (Binance-Kraken, 2-leg pre-funded)',
     'n_legs': 2, 'range_leg_cols': ['binance_btcusdt', 'kraken_btcusdt']},
    {'channel_key': 'xbasis_btcusd_coinbase_kraken', 'basis_col': 'xbasis_btcusd_coinbase_kraken',
     'label_short': 'Cross BTC/USD CB-Kra (2-leg)', 'label_table': 'Cross BTC/USD (Coinbase-Kraken, 2-leg pre-funded)',
     'n_legs': 2, 'range_leg_cols': ['coinbase_btcusd', 'kraken_btcusd']},
]

arb_channel_data = {}
for spec in arb_channel_specs:
    basis_col = spec['basis_col']
    if basis_col not in basis.columns:
        continue
    if any(rc not in ranges.columns for rc in spec['range_leg_cols']):
        continue
    channel_df = pd.DataFrame(index=basis.index)
    channel_df['abs_basis_bps'] = basis[basis_col].abs()
    for i, rc in enumerate(spec['range_leg_cols'], 1):
        channel_df[f'leg_range_{i}_bps'] = ranges[rc] * 10000.0
    channel_df = channel_df.dropna()
    if channel_df.empty: continue
    fee_component = spec['n_legs'] * FEE_BPS_PER_LEG
    leg_cols = [f'leg_range_{i}_bps' for i in range(1, len(spec['range_leg_cols'])+1)]
    channel_df['slippage_cost_bps'] = 0.5 * channel_df[leg_cols].sum(axis=1)
    channel_df['cost_fee_only_bps'] = fee_component
    channel_df['cost_fee_slippage_bps'] = fee_component + channel_df['slippage_cost_bps']
    channel_df['net_fee_only_bps'] = (channel_df['abs_basis_bps'] - channel_df['cost_fee_only_bps']).clip(lower=0.0)
    channel_df['net_fee_slippage_bps'] = (channel_df['abs_basis_bps'] - channel_df['cost_fee_slippage_bps']).clip(lower=0.0)
    arb_channel_data[spec['channel_key']] = {'spec': spec, 'df': channel_df}

# Build summary table
arb_summary = []
cost_variants = [
    ('fee_only_upper', 'Fee-only upper bound', 'cost_fee_only_bps', 'net_fee_only_bps'),
    ('fee_plus_slippage_conservative', 'Fee + slippage conservative bound', 'cost_fee_slippage_bps', 'net_fee_slippage_bps'),
]
for channel_obj in arb_channel_data.values():
    spec = channel_obj['spec']
    cdf = channel_obj['df']
    for regime, (t0, t1) in regimes.items():
        mask = (cdf.index >= t0) & (cdf.index < t1)
        sub = cdf.loc[mask]
        if sub.empty: continue
        for vk, vl, cc, nc in cost_variants:
            profitable = sub['abs_basis_bps'] > sub[cc]
            arb_summary.append({
                'channel': spec['label_table'], 'regime': regime,
                'cost_variant': vk, 'mean_abs_bps': sub['abs_basis_bps'].mean(),
                'pct_profitable': profitable.mean() * 100.0,
                'avg_net_uncond_bps': sub[nc].mean(),
            })

df_arb = pd.DataFrame(arb_summary)
df_arb.to_csv(os.path.join(TABLES_DIR, 'arbitrage_summary.csv'), index=False)

# Compact table for paper
compact_specs_arb = [
    ('USDC/USD (Kraken, 3-leg triangular)', 'Crisis'),
    ('USDC/USD (Kraken, 3-leg triangular)', 'Post-SVB'),
    ('USDT/USD (Kraken, 3-leg triangular)', 'Crisis'),
    ('Cross BTC/USD (Coinbase-Kraken, 2-leg pre-funded)', 'Crisis'),
    ('Cross BTC/USDT (Binance-Kraken, 2-leg pre-funded)', 'Crisis'),
]
compact_rows_arb = []
for channel, reg in compact_specs_arb:
    sub = df_arb[(df_arb['channel'] == channel) & (df_arb['regime'] == reg)]
    if sub.empty: continue
    for cv in ['fee_only_upper', 'fee_plus_slippage_conservative']:
        row = sub[sub['cost_variant'] == cv]
        if row.empty: continue
        r = row.iloc[0]
        compact_rows_arb.append({'Channel': channel.split('(')[0].strip(),
                                 'Regime': reg,
                                 'Cost': 'Fee-only' if 'fee_only' in cv else 'Fee+slip',
                                 '%Profitable': float(r['pct_profitable']),
                                 'AvgNetUncond (bps)': float(r['avg_net_uncond_bps'])})

df_arb_compact = pd.DataFrame(compact_rows_arb)
with open(os.path.join(TABLES_DIR, 'arbitrage_compact.tex'), 'w') as f:
    f.write(df_arb_compact.to_latex(index=False, caption='Arbitrage Profitability by Channel and Regime',
                                     label='tab:arb', column_format='llcrr', float_format='%.2f', escape=True))
print('Arbitrage summary:')
print(df_arb_compact.to_string(index=False))


Arbitrage summary:
       Channel   Regime     Cost  %Profitable  AvgNetUncond (bps)
      USDC/USD   Crisis Fee-only    29.161802            4.834001
      USDC/USD   Crisis Fee+slip     6.724259            0.642646
      USDC/USD Post-SVB Fee-only     9.380788            0.561402
      USDC/USD Post-SVB Fee+slip     1.779531            0.075110
      USDT/USD   Crisis Fee-only     3.756957            0.142555
      USDT/USD   Crisis Fee+slip     0.371058            0.009324
 Cross BTC/USD   Crisis Fee-only    11.319444            0.719758
 Cross BTC/USD   Crisis Fee+slip     2.291667            0.145992
Cross BTC/USDT   Crisis Fee-only    11.850649            0.488596
Cross BTC/USDT   Crisis Fee+slip     1.808905            0.049778


---
# Section 6: Academic Improvements

Six targeted improvements from `src/06_three_fixes.py`:
1. Roll (1984) effective spread + Amihud (2002) ILLIQ (Table 4, Figure 8)
2. Hasbrouck (1995) Information Shares
3. GENIUS Act counterfactual (Table 9)
4. Data coverage table (Table 1)
5. HAC uncertainty intervals (Table 5)
6. Distributional robustness — Chow test, moments, correlation (Table 10)


### 6.1 Roll Spread & Amihud ILLIQ (Table 4, Figure 8)


In [15]:
# Roll (1984) effective spread
def roll_spread_daily(price_col):
    p = prices[price_col].dropna()
    lr = np.log(p / p.shift(1))
    rows = []
    for date, grp in lr.groupby(lr.index.date):
        r = grp.dropna().values
        if len(r) < 15: continue
        cov = np.cov(r[1:], r[:-1])[0, 1]
        rows.append({'date': pd.Timestamp(date),
                     'roll_bps': 2.0 * np.sqrt(-cov) * 10000 if cov < 0 else np.nan})
    if not rows: return pd.Series(dtype=float)
    s = pd.DataFrame(rows).set_index('date')['roll_bps']
    s.index = pd.DatetimeIndex(s.index).tz_localize('UTC')
    return s

# Amihud (2002) ILLIQ
def amihud_daily(price_col, vol_col):
    if vol_col not in volumes.columns: return pd.Series(dtype=float)
    p = prices[price_col]; v = volumes[vol_col]
    lr = np.log(p / p.shift(1)).abs()
    dvol = v * p
    aligned = pd.concat([lr, dvol], axis=1, keys=['abs_ret', 'dvol']).dropna()
    aligned = aligned[aligned['dvol'] > 1.0]
    aligned['illiq'] = aligned['abs_ret'] / aligned['dvol']
    daily = aligned.groupby(aligned.index.date)['illiq'].mean() * 1e6
    s = daily.copy(); s.index = pd.DatetimeIndex(s.index).tz_localize('UTC')
    return s

# Dollar volume depth proxy
def dollar_volume_daily(price_col, vol_col):
    if price_col not in prices.columns or vol_col not in volumes.columns:
        return pd.Series(dtype=float)
    dvol = (prices[price_col] * volumes[vol_col]).dropna()
    if dvol.empty: return pd.Series(dtype=float)
    out = dvol.groupby(dvol.index.date).sum()
    out.index = pd.DatetimeIndex(out.index).tz_localize('UTC')
    return out

PAIRS = {
    'Kraken BTC/USD':   ('kraken_btcusd',   'kraken_btcusd'),
    'Kraken BTC/USDT':  ('kraken_btcusdt',  'kraken_btcusdt'),
    'Kraken BTC/USDC':  ('kraken_btcusdc',  'kraken_btcusdc'),
    'Binance BTC/USDT': ('binance_btcusdt', 'binance_btcusdt'),
    'Coinbase BTC/USD': ('coinbase_btcusd', 'coinbase_btcusd'),
}
PAIR_ORDER = list(PAIRS.keys())

roll_series   = {lbl: roll_spread_daily(pc) for lbl, (pc, _)  in PAIRS.items() if pc in prices.columns}
amihud_series = {lbl: amihud_daily(pc, vc) for lbl, (pc, vc) in PAIRS.items() if pc in prices.columns}
depth_series  = {lbl: dollar_volume_daily(pc, vc) for lbl, (pc, vc) in PAIRS.items() if pc in prices.columns}

def regime_stats_daily(daily_dict):
    rows = []
    for lbl in PAIR_ORDER:
        if lbl not in daily_dict: continue
        s = daily_dict[lbl]
        for reg in REGIME_ORDER:
            if reg == 'Pre-SVB':  mask = s.index < svb_start
            elif reg == 'Crisis': mask = (s.index >= svb_start) & (s.index < svb_end)
            else:                 mask = s.index >= svb_end
            sub = s[mask].dropna()
            rows.append({'Pair': lbl, 'Regime': reg,
                         'mean': round(sub.mean(), 3) if len(sub) else np.nan, 'N': len(sub)})
    return pd.DataFrame(rows)

df_roll   = regime_stats_daily(roll_series)
df_amihud = regime_stats_daily(amihud_series)

# Save Roll + Amihud table
roll_pivot   = df_roll.pivot(index='Pair', columns='Regime', values='mean')[REGIME_ORDER].reindex(PAIR_ORDER)
amihud_pivot = df_amihud.pivot(index='Pair', columns='Regime', values='mean')[REGIME_ORDER].reindex(PAIR_ORDER)
print('Roll Spread (bps):')
print(roll_pivot)
print('\nAmihud ILLIQ (x1e-6):')
print(amihud_pivot)

# Depth proxy
depth_rows = []
for lbl in PAIR_ORDER:
    if lbl not in depth_series: continue
    s = depth_series[lbl].dropna()
    for reg in REGIME_ORDER:
        if reg == 'Pre-SVB':  mask = s.index < svb_start
        elif reg == 'Crisis': mask = (s.index >= svb_start) & (s.index < svb_end)
        else:                 mask = s.index >= svb_end
        sub = s[mask].dropna()
        depth_rows.append({'Pair': lbl, 'Regime': reg,
                           'median_usd_mm': (sub.median() / 1e6) if len(sub) else np.nan})
df_depth = pd.DataFrame(depth_rows)
print('\nDollar-volume depth proxy (median $MM/day):')
print(df_depth.pivot(index='Pair', columns='Regime', values='median_usd_mm')[REGIME_ORDER].reindex(PAIR_ORDER))


Roll Spread (bps):
Regime            Pre-SVB  Crisis  Post-SVB
Pair                                       
Kraken BTC/USD      1.256     NaN       NaN
Kraken BTC/USDT     3.390   2.852     2.674
Kraken BTC/USDC     1.095  25.640     2.072
Binance BTC/USDT    1.555   0.669     3.633
Coinbase BTC/USD    1.882   2.488     3.492

Amihud ILLIQ (x1e-6):
Regime            Pre-SVB  Crisis  Post-SVB
Pair                                       
Kraken BTC/USD      0.232   0.050     0.068
Kraken BTC/USDT     8.299   7.266     9.074
Kraken BTC/USDC    31.970  13.282    23.530
Binance BTC/USDT    0.163   0.461     0.591
Coinbase BTC/USD    0.005   0.003     0.003

Dollar-volume depth proxy (median $MM/day):
Regime               Pre-SVB      Crisis    Post-SVB
Pair                                                
Kraken BTC/USD     59.498447  160.360359  203.769439
Kraken BTC/USDT     5.552533   13.840601   17.369753
Kraken BTC/USDC     2.555887   15.654839    5.519398
Binance BTC/USDT   52.444511   7

### 6.2 Hasbrouck Information Shares


In [16]:
# Hasbrouck (1995) Information Shares
def hasbrouck_is_bounds(vecm_fit, mkt1_name='Market 1', mkt2_name='Market 2'):
    alpha = vecm_fit.alpha[:, 0].astype(float)
    sigma = vecm_fit.sigma_u.astype(float)
    psi = np.array([-alpha[1], alpha[0]])
    results = {}
    for ordering, (i, j) in [('12', (0, 1)), ('21', (1, 0))]:
        idx = [i, j]
        sig_p = sigma[np.ix_(idx, idx)]
        try:
            M = np.linalg.cholesky(sig_p)
        except np.linalg.LinAlgError:
            results[ordering] = (np.nan, np.nan); continue
        psi_p = psi[idx]
        f = psi_p @ M
        denom = float(f @ f)
        if denom < 1e-20:
            results[ordering] = (np.nan, np.nan); continue
        IS = [0.0, 0.0]; IS[i] = float(f[0]**2 / denom); IS[j] = float(f[1]**2 / denom)
        results[ordering] = tuple(IS)
    vals_mkt1 = [v[0] for v in results.values() if not np.isnan(v[0])]
    vals_mkt2 = [v[1] for v in results.values() if not np.isnan(v[1])]
    IS1_lo = min(vals_mkt1) if vals_mkt1 else np.nan
    IS1_hi = max(vals_mkt1) if vals_mkt1 else np.nan
    IS2_lo = min(vals_mkt2) if vals_mkt2 else np.nan
    IS2_hi = max(vals_mkt2) if vals_mkt2 else np.nan
    return {
        f'IS_{mkt1_name}_lower': IS1_lo, f'IS_{mkt1_name}_upper': IS1_hi,
        f'IS_{mkt1_name}_midpoint': 0.5*(IS1_lo+IS1_hi),
        f'IS_{mkt2_name}_lower': IS2_lo, f'IS_{mkt2_name}_upper': IS2_hi,
        f'IS_{mkt2_name}_midpoint': 0.5*(IS2_lo+IS2_hi),
        'alpha_mkt1': float(alpha[0]), 'alpha_mkt2': float(alpha[1]),
    }

# Fit VECM for Kraken BTC/USD vs BTC/USDT
c1, c2 = 'kraken_btcusd', 'kraken_btcusdt'
ff_mask_is = price_ff_flags[[c1, c2]].any(axis=1)
p_log = np.log(prices[[c1, c2]]).copy()
p_log[ff_mask_is] = np.nan
df_levels_is = p_log.dropna()

sel_is = select_order(df_levels_is, maxlags=15, deterministic='ci')
p_bic_is = sel_is.bic
p_aic_is = sel_is.aic
p_used_is = p_bic_is if p_bic_is is not None else (p_aic_is if p_aic_is is not None else 2)
p_used_is = max(1, int(p_used_is))
k_diff_is = p_used_is - 1

joh_is = coint_johansen(df_levels_is.values, det_order=0, k_ar_diff=k_diff_is)
rank_is = min(int(np.sum(joh_is.lr1 > joh_is.cvt[:, 1])), 1)

if rank_is >= 1:
    vecm_is = VECM(df_levels_is, k_ar_diff=k_diff_is, coint_rank=rank_is, deterministic='ci').fit()
    is_d = hasbrouck_is_bounds(vecm_is, mkt1_name='USD', mkt2_name='USDT')
    print(f'Hasbrouck IS for BTC/USD: [{is_d["IS_USD_lower"]:.3f}, {is_d["IS_USD_upper"]:.3f}], midpoint={is_d["IS_USD_midpoint"]:.3f}')
    print(f'Hasbrouck IS for BTC/USDT: [{is_d["IS_USDT_lower"]:.3f}, {is_d["IS_USDT_upper"]:.3f}], midpoint={is_d["IS_USDT_midpoint"]:.3f}')
else:
    print('rank=0, no cointegration found for Hasbrouck IS')


Hasbrouck IS for BTC/USD: [0.686, 0.772], midpoint=0.729
Hasbrouck IS for BTC/USDT: [0.228, 0.314], midpoint=0.271


### 6.3 Data Coverage Table (Table 1)


In [17]:
# Data coverage table
coverage_specs = [
    ('kraken_btcusd', 'Kraken BTC/USD'), ('kraken_btcusdt', 'Kraken BTC/USDT'),
    ('kraken_btcusdc', 'Kraken BTC/USDC'), ('kraken_usdcusd', 'Kraken USDC/USD'),
    ('kraken_usdtusd', 'Kraken USDT/USD'), ('coinbase_btcusd', 'Coinbase BTC/USD'),
    ('coinbase_btcusdt', 'Coinbase BTC/USDT'), ('coinbase_usdtusd', 'Coinbase USDT/USD'),
    ('binance_btcusdt', 'Binance BTC/USDT'), ('binance_btcusdc', 'Binance BTC/USDC'),
]
cov_rows = []
for col, label in coverage_specs:
    if col not in prices.columns: continue
    s = prices[col]
    ff = price_ff_flags[col] if col in price_ff_flags.columns else pd.Series(False, index=prices.index)
    cov_rows.append({
        'Pair': label,
        'Coverage Overall (%)': s.notna().mean() * 100,
        'Coverage Pre-SVB (%)': s.loc[m_pre].notna().mean() * 100,
        'Coverage Crisis (%)': s.loc[m_cri].notna().mean() * 100,
        'Coverage Post-SVB (%)': s.loc[m_post].notna().mean() * 100,
        'Forward-Fill Share Overall (%)': ff.reindex(prices.index).fillna(False).mean() * 100,
    })
df_cov = pd.DataFrame(cov_rows)
df_cov.to_csv(os.path.join(TABLES_DIR, 'data_coverage_core.csv'), index=False)
print('Data coverage table:')
print(df_cov.to_string(index=False))


Data coverage table:
             Pair  Coverage Overall (%)  Coverage Pre-SVB (%)  Coverage Crisis (%)  Coverage Post-SVB (%)  Forward-Fill Share Overall (%)
   Kraken BTC/USD             99.976852            100.000000           100.000000              99.945988                        0.178571
  Kraken BTC/USDT             98.439153             97.361111            99.814815              99.058642                       30.839947
  Kraken BTC/USDC             94.050926             89.621914            99.791667              96.566358                       42.070106
  Kraken USDC/USD             97.800926             96.026235            99.351852              99.058642                       28.574735
  Kraken USDT/USD             99.976852            100.000000           100.000000              99.945988                        0.906085
 Coinbase BTC/USD             99.100529             97.901235           100.000000             100.000000                        0.059524
Coinbase BTC/

### 6.4 GENIUS Act Counterfactual (Table 9)


In [18]:
# GENIUS Act counterfactual
d_col_g = 'dispersion_usdc_kraken'
def d_stats_g(col, t0, t1):
    mask = (basis.index >= t0) & (basis.index < t1)
    s = basis.loc[mask, col].dropna()
    return {'mean': s.mean(), 'std': s.std(), 'p99': s.quantile(0.99), 'N': len(s)}

pre_d = d_stats_g(d_col_g, prices.index.min(), svb_start)
cri_d = d_stats_g(d_col_g, svb_start, svb_end)

shock_component = cri_d['mean'] - pre_d['mean']
mitigation_grid = [0.00, 0.25, 0.50, 0.75, 1.00]
scenario_map = {0.00: 'Observed baseline', 0.25: 'Low mitigation',
                0.50: 'Moderate mitigation', 0.75: 'High mitigation',
                1.00: 'Full-mitigation lower bound'}

cf_rows = []
for m in mitigation_grid:
    implied_mean = pre_d['mean'] + (1.0 - m) * shock_component
    cf_rows.append({'Scenario': scenario_map[m],
                    'Assumed mitigation (%)': int(round(m * 100)),
                    'Implied D_t mean (bps)': round(implied_mean, 1),
                    'Reduction vs observed (bps)': round(cri_d['mean'] - implied_mean, 1)})

df_cf = pd.DataFrame(cf_rows)
df_cf.to_csv(os.path.join(TABLES_DIR, 'genius_counterfactual.csv'), index=False)
print('GENIUS Act counterfactual scenarios:')
print(df_cf.to_string(index=False))


GENIUS Act counterfactual scenarios:
                   Scenario  Assumed mitigation (%)  Implied D_t mean (bps)  Reduction vs observed (bps)
          Observed baseline                       0                   320.0                          0.0
             Low mitigation                      25                   240.0                         80.0
        Moderate mitigation                      50                   160.0                        159.9
            High mitigation                      75                    80.1                        239.9
Full-mitigation lower bound                     100                     0.1                        319.9


### 6.5 HAC Uncertainty Intervals (Table 5)


In [19]:
# HAC uncertainty for headline means
def hac_mean_ci(series, maxlags=60):
    s = series.dropna()
    n = int(len(s))
    if n < 10: return {'mean': np.nan, 'se': np.nan, 'ci_lo': np.nan, 'ci_hi': np.nan, 'N': n}
    lag_use = min(maxlags, max(1, n // 10))
    model = sm.OLS(s.values, np.ones((n, 1))).fit(cov_type='HAC', cov_kwds={'maxlags': lag_use})
    mean = float(model.params[0]); se = float(model.bse[0])
    return {'mean': mean, 'se': se, 'ci_lo': mean - 1.96*se, 'ci_hi': mean + 1.96*se, 'N': n}

hac_specs = [
    ('USDC D_t (Kraken, crisis)', basis.loc[m_cri, 'dispersion_usdc_kraken']),
    ('USDC B_t (Kraken, crisis)', basis.loc[m_cri, 'basis_usdc_kraken']),
    ('USDT premium (Kraken, crisis)', (prices.loc[m_cri, 'kraken_usdtusd'] - 1.0) * 10000),
    ('USDT premium (Coinbase, crisis)', (prices.loc[m_cri, 'coinbase_usdtusd'] - 1.0) * 10000),
    ('BTC/USDT basis (BN-KR, calm)', basis.loc[m_calm, 'xbasis_btcusdt_binance_kraken']),
]
hac_rows = []
for metric, ser in hac_specs:
    out = hac_mean_ci(ser, maxlags=60)
    hac_rows.append({'Metric': metric, 'Mean (bps)': round(out['mean'], 3),
                     'HAC SE': round(out['se'], 3),
                     '95% CI': f"[{out['ci_lo']:.3f}, {out['ci_hi']:.3f}]", 'N': out['N']})
df_hac = pd.DataFrame(hac_rows)
df_hac.to_csv(os.path.join(TABLES_DIR, 'hac_headline_metrics.csv'), index=False)
print('HAC uncertainty intervals:')
print(df_hac.to_string(index=False))


HAC uncertainty intervals:
                         Metric  Mean (bps)  HAC SE             95% CI     N
      USDC D_t (Kraken, crisis)     319.974  37.908 [245.674, 394.273]  4311
      USDC B_t (Kraken, crisis)       5.271   0.873     [3.559, 6.983]  4283
  USDT premium (Kraken, crisis)      58.114   5.467   [47.399, 68.830]  4320
USDT premium (Coinbase, crisis)      59.680   5.405   [49.087, 70.273]  4320
   BTC/USDT basis (BN-KR, calm)       0.157   0.066     [0.028, 0.286] 25456


### 6.6 Distributional Robustness — Chow, Moments, Correlation (Table 10)


In [20]:
# Chow structural-break test
disp_usdc = basis['dispersion_usdc_kraken'].dropna()
n_total = len(disp_usdc)
pre_break = disp_usdc[disp_usdc.index < svb_start]
post_break = disp_usdc[disp_usdc.index >= svb_start]
n1, n2 = len(pre_break), len(post_break)
rss_full = float(sm.OLS(disp_usdc.values, np.ones((n_total, 1))).fit().ssr)
rss1 = float(sm.OLS(pre_break.values, np.ones((n1, 1))).fit().ssr)
rss2 = float(sm.OLS(post_break.values, np.ones((n2, 1))).fit().ssr)
k = 1
chow_F = ((rss_full - rss1 - rss2) / k) / ((rss1 + rss2) / (n_total - 2*k))
chow_p = 1.0 - sp_stats.f.cdf(chow_F, k, n_total - 2*k)
print(f'Chow break test for USDC D_t at SVB start: F={chow_F:.1f}, p={chow_p:.2e}')

# Higher moments by regime
moment_rows = []
for label, col in [('USDC', 'basis_usdc_kraken'), ('USDT', 'basis_usdt_kraken')]:
    for regime, mask in [('Pre-SVB', m_pre), ('Crisis', m_cri), ('Post-SVB', m_post)]:
        s = basis.loc[mask, col].dropna()
        moment_rows.append({
            'Channel': label, 'Regime': regime, 'N': len(s),
            'Mean': float(s.mean()), 'Std': float(s.std()),
            'Skewness': float(sp_stats.skew(s)),
            'Excess Kurtosis': float(sp_stats.kurtosis(s)),
        })
df_moments = pd.DataFrame(moment_rows)
df_moments.to_csv(os.path.join(TABLES_DIR, 'distributional_robustness.csv'), index=False)
print('\nDistributional robustness:')
print(df_moments.to_string(index=False))

# Cross-stablecoin correlation
corr_rows = []
for regime, mask in [('Pre-SVB', m_pre), ('Crisis', m_cri), ('Post-SVB', m_post)]:
    usdc_s = basis.loc[mask, 'basis_usdc_kraken'].dropna()
    usdt_s = basis.loc[mask, 'basis_usdt_kraken'].dropna()
    common = usdc_s.index.intersection(usdt_s.index)
    rho = float(usdc_s.loc[common].corr(usdt_s.loc[common]))
    corr_rows.append({'Regime': regime, 'Corr(B_USDC, B_USDT)': rho, 'N': len(common)})
df_corr_regime = pd.DataFrame(corr_rows)
print('\nCross-stablecoin B_t correlation by regime:')
print(df_corr_regime.to_string(index=False))


Chow break test for USDC D_t at SVB start: F=2088.7, p=1.11e-16

Distributional robustness:
Channel   Regime     N      Mean       Std  Skewness  Excess Kurtosis
   USDC  Pre-SVB 11197 -0.323817  4.995510  0.002553         3.775833
   USDC   Crisis  4283  5.271050 21.208694  1.011576        12.915226
   USDC Post-SVB 12420  0.453536  9.055213  0.301257         2.799943
   USDT  Pre-SVB 12618 -0.596262  4.329247 -0.364901         5.229107
   USDT   Crisis  4312  0.680782  6.566265 -0.111354         1.376329
   USDT Post-SVB 12838 -1.226566  6.471544 -0.029900         1.953393

Cross-stablecoin B_t correlation by regime:
  Regime  Corr(B_USDC, B_USDT)     N
 Pre-SVB              0.439997 10948
  Crisis              0.155526  4275
Post-SVB              0.377213 12340


---
# Section 7: Novel Mathematical Contributions

From `src/10_novel_contributions.py`:
1. **Contagion Intensity Model**: Coupled OU with lagged peg deviation (Table 3)
2. **Half-Life Ratio Bootstrap**: Formal CI for the ~940x gap
3. **Asymmetry Test**: Differential lambda for peg discounts vs premiums


In [21]:
# ====================================================================
# 1. CONTAGION INTENSITY MODEL
# ====================================================================
print("=" * 70)
print("1. CONTAGION INTENSITY MODEL")
print("=" * 70)

def estimate_contagion(B_col, S_col, regime_name, t0, t1, ff_filter=False):
    """
    Estimate: Delta B_t = a + beta * B_{t-1} + lambda * S_{t-1} + eps_t
    with HAC standard errors (Newey-West, 60 lags).

    Returns dict of results.
    """
    mask = (basis.index >= t0) & (basis.index < t1)

    df = pd.DataFrame({
        'B': basis.loc[mask, B_col],
        'S': basis.loc[mask, S_col],
    }).dropna()

    if ff_filter and B_col in basis_ff_flags.columns:
        ff_B = basis_ff_flags.loc[mask, B_col].reindex(df.index).fillna(False)
        df = df[~ff_B]

    if len(df) < 100:
        return None

    # Construct lagged variables
    df['B_lag'] = df['B'].shift(1)
    df['S_lag'] = df['S'].shift(1)
    df['dB'] = df['B'] - df['B_lag']
    df = df.dropna()

    if len(df) < 100:
        return None

    # OLS with HAC
    X = sm.add_constant(df[['B_lag', 'S_lag']])
    y = df['dB']
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 60})

    beta = float(model.params['B_lag'])
    lam = float(model.params['S_lag'])
    lam_se = float(model.bse['S_lag'])
    lam_p = float(model.pvalues['S_lag'])
    beta_p = float(model.pvalues['B_lag'])
    rho_implied = 1.0 + beta

    # Half-life from implied rho (with lambda controlled for)
    hl = half_life_from_rho(rho_implied, dt_minutes=1.0) if 0 < rho_implied < 1 else np.nan

    return {
        'regime': regime_name,
        'n_obs': int(model.nobs),
        'beta': beta,
        'beta_p': beta_p,
        'rho_implied': rho_implied,
        'half_life_min': hl,
        'lambda': lam,
        'lambda_se': lam_se,
        'lambda_t': lam / lam_se if lam_se > 0 else np.nan,
        'lambda_p': lam_p,
        'R2': float(model.rsquared),
        'const': float(model.params['const']),
        'const_p': float(model.pvalues['const']),
    }

# Primary: USDC channel
print("\n--- USDC Channel (Kraken) ---")
usdc_results = {}
for regime, (t0, t1) in regimes.items():
    res = estimate_contagion('basis_usdc_kraken', 'usdc_peg_dev_kraken', regime, t0, t1)
    if res:
        usdc_results[regime] = res
        sig = '***' if res['lambda_p'] < 0.001 else ('**' if res['lambda_p'] < 0.01 else ('*' if res['lambda_p'] < 0.05 else ''))
        print(f"  {regime:10s}: lambda={res['lambda']:+.5f} (SE={res['lambda_se']:.5f}, t={res['lambda_t']:.2f}, p={res['lambda_p']:.4f}) {sig}")
        print(f"             beta={res['beta']:+.5f} (p={res['beta_p']:.4f}), rho={res['rho_implied']:.4f}, HL={res['half_life_min']:.2f} min, R2={res['R2']:.4f}, N={res['n_obs']}")

# Secondary: USDT channel
print("\n--- USDT Channel (Kraken) ---")
usdt_results = {}
for regime, (t0, t1) in regimes.items():
    res = estimate_contagion('basis_usdt_kraken', 'usdt_peg_dev_kraken', regime, t0, t1)
    if res:
        usdt_results[regime] = res
        sig = '***' if res['lambda_p'] < 0.001 else ('**' if res['lambda_p'] < 0.01 else ('*' if res['lambda_p'] < 0.05 else ''))
        print(f"  {regime:10s}: lambda={res['lambda']:+.5f} (SE={res['lambda_se']:.5f}, t={res['lambda_t']:.2f}, p={res['lambda_p']:.4f}) {sig}")
        print(f"             beta={res['beta']:+.5f} (p={res['beta_p']:.4f}), rho={res['rho_implied']:.4f}, HL={res['half_life_min']:.2f} min, R2={res['R2']:.4f}, N={res['n_obs']}")

# Robustness: no-FF filter for USDC crisis
print("\n--- USDC Crisis No-FF Robustness ---")
res_noff = estimate_contagion('basis_usdc_kraken', 'usdc_peg_dev_kraken', 'Crisis', svb_start, svb_end, ff_filter=True)
if res_noff:
    print(f"  Crisis(no-FF): lambda={res_noff['lambda']:+.5f} (SE={res_noff['lambda_se']:.5f}, p={res_noff['lambda_p']:.4f}), N={res_noff['n_obs']}")

# Robustness: 5-minute sampling for USDC crisis
print("\n--- USDC Crisis 5-min Robustness ---")
mask_crisis = (basis.index >= svb_start) & (basis.index < svb_end)
B_5m = basis.loc[mask_crisis, 'basis_usdc_kraken'].resample('5min').last()
S_5m = basis.loc[mask_crisis, 'usdc_peg_dev_kraken'].resample('5min').last()
df_5m = pd.DataFrame({'B': B_5m, 'S': S_5m}).dropna()
df_5m['B_lag'] = df_5m['B'].shift(1)
df_5m['S_lag'] = df_5m['S'].shift(1)
df_5m['dB'] = df_5m['B'] - df_5m['B_lag']
df_5m = df_5m.dropna()
if len(df_5m) >= 50:
    X_5m = sm.add_constant(df_5m[['B_lag', 'S_lag']])
    m_5m = sm.OLS(df_5m['dB'], X_5m).fit(cov_type='HAC', cov_kwds={'maxlags': 12})
    print(f"  Crisis(5m): lambda={float(m_5m.params['S_lag']):+.5f} (SE={float(m_5m.bse['S_lag']):.5f}, p={float(m_5m.pvalues['S_lag']):.4f}), N={int(m_5m.nobs)}")


# ====================================================================
# 2. HALF-LIFE RATIO BOOTSTRAP CI
# ====================================================================
print("\n" + "=" * 70)
print("2. HALF-LIFE RATIO BOOTSTRAP CI")
print("=" * 70)

def estimate_rho_from_series(series, dt_minutes=1.0):
    """Estimate AR(1) rho from a series via ECM regression."""
    s = series.dropna().astype(float)
    if len(s) < 100:
        return np.nan, len(s)
    reg = pd.DataFrame({'x': s})
    reg['x_lag'] = reg['x'].shift(1)
    reg['dx'] = reg['x'] - reg['x_lag']
    reg = reg.dropna()
    if len(reg) < 100:
        return np.nan, len(reg)
    X = sm.add_constant(reg['x_lag'])
    model = sm.OLS(reg['dx'], X).fit()
    beta = float(model.params.iloc[1])
    rho = 1.0 + beta
    return rho, len(reg)


# Point estimates for crisis
B_crisis = basis.loc[mask_crisis, 'basis_usdc_kraken'].dropna()
S_crisis = basis.loc[mask_crisis, 'usdc_peg_dev_kraken'].dropna()

rho_B, n_B = estimate_rho_from_series(B_crisis)
rho_S, n_S = estimate_rho_from_series(S_crisis)

hl_B = half_life_from_rho(rho_B, 1.0)
hl_S = half_life_from_rho(rho_S, 1.0) if 0 < rho_S < 1 else np.inf
R_point = hl_S / hl_B if np.isfinite(hl_B) and hl_B > 0 else np.nan

print(f"\nPoint estimates (crisis):")
print(f"  rho_B = {rho_B:.6f}, HL_B = {hl_B:.2f} min")
print(f"  rho_S = {rho_S:.6f}, HL_S = {hl_S:.2f} min")
print(f"  R = HL_S / HL_B = {R_point:.1f}")

# Block bootstrap
np.random.seed(42)
BLOCK_LEN = 60  # 1 hour blocks
N_BOOT = 5000

def block_bootstrap_rho(series, block_len, n_boot):
    """
    Block bootstrap for AR(1) rho estimate.
    Returns array of bootstrap rho estimates.
    """
    s = series.dropna().values
    n = len(s)
    n_blocks = int(np.ceil(n / block_len))
    rhos = np.full(n_boot, np.nan)

    for b in range(n_boot):
        # Draw random block starts
        starts = np.random.randint(0, n - block_len + 1, size=n_blocks)
        # Concatenate blocks
        boot_sample = np.concatenate([s[st:st+block_len] for st in starts])[:n]

        # Estimate rho
        x = boot_sample[:-1]
        dx = np.diff(boot_sample)
        if len(x) < 50:
            continue
        X = np.column_stack([np.ones(len(x)), x])
        try:
            beta = np.linalg.lstsq(X, dx, rcond=None)[0][1]
            rhos[b] = 1.0 + beta
        except:
            pass

    return rhos

print("\nRunning block bootstrap (5000 replications, block=60 min)...")

rhos_B_boot = block_bootstrap_rho(B_crisis, BLOCK_LEN, N_BOOT)
rhos_S_boot = block_bootstrap_rho(S_crisis, BLOCK_LEN, N_BOOT)

# Compute half-life ratios
hls_B_boot = np.array([half_life_from_rho(r, 1.0) if 0 < r < 1 else np.nan for r in rhos_B_boot])
hls_S_boot = np.array([
    half_life_from_rho(r, 1.0) if 0 < r < 1 else np.inf
    for r in rhos_S_boot
])

# For ratio: handle cases where S is unit root (hl_S = inf)
# Ratio R = hl_S / hl_B
valid_mask = np.isfinite(hls_B_boot) & (hls_B_boot > 0)
R_boot = np.where(
    valid_mask & np.isfinite(hls_S_boot),
    hls_S_boot / hls_B_boot,
    np.where(valid_mask, np.inf, np.nan)  # inf if S is unit root
)

# For CI: use finite ratios
R_finite = R_boot[np.isfinite(R_boot) & ~np.isnan(R_boot)]
pct_infinite = np.sum(np.isinf(R_boot[~np.isnan(R_boot)])) / np.sum(~np.isnan(R_boot)) * 100

print(f"\nBootstrap results:")
print(f"  Valid bootstrap ratios (finite): {len(R_finite)} / {N_BOOT}")
print(f"  Pct with R=inf (peg unit root): {pct_infinite:.1f}%")

if len(R_finite) > 100:
    R_median = np.median(R_finite)
    R_ci_lo = np.percentile(R_finite, 2.5)
    R_ci_hi = np.percentile(R_finite, 97.5)
    R_ci_lo_90 = np.percentile(R_finite, 5.0)
    # One-sided test: fraction of bootstrap R <= 1
    pval_R1 = np.mean(R_finite <= 1.0)
    # More conservative: fraction of all bootstrap R (including inf) <= 1
    R_all_valid = R_boot[~np.isnan(R_boot)]
    pval_R1_all = np.mean(R_all_valid <= 1.0)

    print(f"  R median (finite): {R_median:.0f}")
    print(f"  R 95% CI (finite): [{R_ci_lo:.0f}, {R_ci_hi:.0f}]")
    print(f"  R 5th percentile (one-sided lower bound): {R_ci_lo_90:.0f}")
    print(f"  p-value for H0: R <= 1 (finite only): {pval_R1:.6f}")
    print(f"  p-value for H0: R <= 1 (all, inf=large): {pval_R1_all:.6f}")

    # Also report hl_B bootstrap CI
    hls_B_finite = hls_B_boot[np.isfinite(hls_B_boot)]
    print(f"\n  HL_B bootstrap: median={np.median(hls_B_finite):.2f}, 95% CI=[{np.percentile(hls_B_finite, 2.5):.2f}, {np.percentile(hls_B_finite, 97.5):.2f}]")

    # hl_S: many infinite, report fraction
    hls_S_finite = hls_S_boot[np.isfinite(hls_S_boot)]
    if len(hls_S_finite) > 10:
        print(f"  HL_S bootstrap (finite only): median={np.median(hls_S_finite):.0f}, 95% CI=[{np.percentile(hls_S_finite, 2.5):.0f}, {np.percentile(hls_S_finite, 97.5):.0f}]")


# ====================================================================
# 3. ASYMMETRY TEST
# ====================================================================
print("\n" + "=" * 70)
print("3. ASYMMETRY TEST (Conditional on lambda significant)")
print("=" * 70)

# Check if crisis lambda is significant
crisis_lam_p = usdc_results.get('Crisis', {}).get('lambda_p', 1.0)
print(f"\nCrisis lambda p-value: {crisis_lam_p:.6f}")

if crisis_lam_p < 0.05:
    print("Lambda is significant -> proceeding with asymmetry test\n")

    # S_t < 0 means USDC below par (discount/de-peg)
    # S_t > 0 means USDC above par (premium)
    # During crisis, S_t is predominantly negative (USDC de-pegged below $1)

    for regime, (t0, t1) in regimes.items():
        mask = (basis.index >= t0) & (basis.index < t1)
        df = pd.DataFrame({
            'B': basis.loc[mask, 'basis_usdc_kraken'],
            'S': basis.loc[mask, 'usdc_peg_dev_kraken'],
        }).dropna()

        if len(df) < 100:
            continue

        df['B_lag'] = df['B'].shift(1)
        df['S_lag'] = df['S'].shift(1)
        df['dB'] = df['B'] - df['B_lag']

        # Asymmetric decomposition
        # S_lag_neg: peg discount (S < 0, i.e., below $1) -- the de-peg direction
        # S_lag_pos: peg premium (S > 0, i.e., above $1)
        df['S_lag_neg'] = df['S_lag'].clip(upper=0)  # negative values (de-peg)
        df['S_lag_pos'] = df['S_lag'].clip(lower=0)  # positive values (premium)

        df = df.dropna()
        if len(df) < 100:
            continue

        X = sm.add_constant(df[['B_lag', 'S_lag_neg', 'S_lag_pos']])
        y = df['dB']
        model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 60})

        lam_neg = float(model.params['S_lag_neg'])
        lam_pos = float(model.params['S_lag_pos'])
        lam_neg_p = float(model.pvalues['S_lag_neg'])
        lam_pos_p = float(model.pvalues['S_lag_pos'])
        lam_neg_se = float(model.bse['S_lag_neg'])
        lam_pos_se = float(model.bse['S_lag_pos'])

        # Wald test for lambda_neg = lambda_pos
        R_mat = np.array([[0, 0, 1, -1]])  # test: S_lag_neg - S_lag_pos = 0
        try:
            wald = model.wald_test(R_mat)
            wald_F = float(wald.statistic[0][0])
            wald_p = float(wald.pvalue)
        except:
            wald_F = np.nan
            wald_p = np.nan

        print(f"  {regime:10s}:")
        print(f"    lambda_neg (de-peg) = {lam_neg:+.5f} (SE={lam_neg_se:.5f}, p={lam_neg_p:.4f})")
        print(f"    lambda_pos (premium)= {lam_pos:+.5f} (SE={lam_pos_se:.5f}, p={lam_pos_p:.4f})")
        print(f"    Wald test (lambda_neg = lambda_pos): F={wald_F:.2f}, p={wald_p:.4f}")
        print(f"    N={int(model.nobs)}, R2={model.rsquared:.4f}")

        # Count observations in each regime
        n_neg = (df['S_lag'] < 0).sum()
        n_pos = (df['S_lag'] >= 0).sum()
        print(f"    Obs with S<0: {n_neg}, S>=0: {n_pos}")
else:
    print("Lambda NOT significant -> skipping asymmetry test")


# ====================================================================
# SUMMARY TABLE FOR LATEX
# ====================================================================
print("\n" + "=" * 70)
print("SUMMARY: LaTeX-ready numbers")
print("=" * 70)

print("\n--- Contagion Intensity Table ---")
for ch_name, ch_results in [('USDC', usdc_results), ('USDT', usdt_results)]:
    print(f"\n{ch_name}:")
    for regime in ['Pre-SVB', 'Crisis', 'Post-SVB']:
        r = ch_results.get(regime)
        if r:
            sig = '***' if r['lambda_p'] < 0.001 else ('**' if r['lambda_p'] < 0.01 else ('*' if r['lambda_p'] < 0.05 else ''))
            p_str = '<0.001' if r['lambda_p'] < 0.001 else f"{r['lambda_p']:.3f}"
            print(f"  {regime}: lambda={r['lambda']:+.4f}{sig}  SE={r['lambda_se']:.4f}  p={p_str}  R2={r['R2']:.3f}  N={r['n_obs']}")


1. CONTAGION INTENSITY MODEL

--- USDC Channel (Kraken) ---
  Pre-SVB   : lambda=-0.00174 (SE=0.09058, t=-0.02, p=0.9846) 
             beta=-0.50557 (p=0.0000), rho=0.4944, HL=0.98 min, R2=0.2528, N=11196
  Crisis    : lambda=-0.00866 (SE=0.00181, t=-4.77, p=0.0000) ***
             beta=-0.69975 (p=0.0000), rho=0.3003, HL=0.58 min, R2=0.3510, N=4282
  Post-SVB  : lambda=-0.01896 (SE=0.00693, t=-2.73, p=0.0062) **
             beta=-0.58518 (p=0.0000), rho=0.4148, HL=0.79 min, R2=0.2927, N=12418

--- USDT Channel (Kraken) ---
  Pre-SVB   : lambda=-0.07624 (SE=0.03199, t=-2.38, p=0.0172) *
             beta=-0.54494 (p=0.0000), rho=0.4551, HL=0.88 min, R2=0.2733, N=12617
  Crisis    : lambda=+0.01876 (SE=0.00359, t=5.23, p=0.0000) ***
             beta=-0.54639 (p=0.0000), rho=0.4536, HL=0.88 min, R2=0.2723, N=4311
  Post-SVB  : lambda=+0.02321 (SE=0.00828, t=2.80, p=0.0050) **
             beta=-0.70739 (p=0.0000), rho=0.2926, HL=0.56 min, R2=0.3536, N=12836

--- USDC Crisis No-FF Rob

---
# Section 8: Column Figures

All 14 single-column figures for the paper, from `src/08_column_figures.py`.
These use the Paul Tol colour-blind-safe palette and 3.4" width for single-column layout.

| # | Figure | Label |
|---|--------|-------|
| 1 | Stablecoin Peg Deviation | fig:peg |
| 2 | $D_t$ vs $B_t$ Decomposition | fig:decomp |
| 3 | Substitution Scatter | fig:substitution |
| 4 | Half-Life Robustness | fig:halflife_robust |
| 5 | Two-Layer Persistence | fig:twolayer |
| 6 | SVB Crisis Zoom | fig:crisiszoom |
| 7 | Cross-Exchange Basis | fig:xbasis |
| 8 | Liquidity Roll+Amihud | fig:liq |
| 9 | Volume Share | fig:volshare |
| 10 | Realized Volatility | fig:realvol |
| 11 | Tail Blowout KDE | fig:tailblowout |
| 12 | Correlation Heatmap | fig:corrheat |
| 13 | VAR IRF | fig:irf |
| 14 | Arbitrage After Fees | fig:arbfees |


In [22]:
# Column figure style and palette setup
# (Data already loaded above)

# ── Colour palette (Paul Tol "bright" – colour-blind safe) ────────────────────
CB_BLUE   = '#4477AA'
CB_RED    = '#EE6677'
CB_GRAY   = '#BBBBBB'
CB_ORANGE = '#CCBB44'
CB_GREEN  = '#228833'
CB_PURPLE = '#AA3377'
CB_CYAN   = '#66CCEE'

REGIME_COLORS = {'Pre-SVB': CB_BLUE, 'Crisis': CB_RED, 'Post-SVB': CB_GREEN}

# ── Single-column RC (applies to EVERY figure) ────────────────────────────────
COL_RC = {
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset':  'stix',
    'axes.facecolor':    'white',
    'figure.facecolor':  'white',
    'axes.grid':         False,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.linewidth':    0.6,
    'font.size':         7,
    'axes.titlesize':    8,
    'axes.labelsize':    7.5,
    'legend.fontsize':   6.5,
    'xtick.labelsize':   6.5,
    'ytick.labelsize':   6.5,
    'xtick.direction':   'out',
    'ytick.direction':   'out',
    'lines.linewidth':   0.65,
    'xtick.major.size':  2.5,
    'ytick.major.size':  2.5,
}

# ── Helpers ───────────────────────────────────────────────────────────────────
def save(fig, name):
    path = os.path.join(FIGURES_COL, name)
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  saved → {name}")

def shade_crisis(ax):
    ax.axvspan(svb_start, svb_end, alpha=0.15, color='red', zorder=0)

def fmt_date(ax, fmt='%b %d', rotation=30):
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=rotation, ha='right')


# ══════════════════════════════════════════════════════════════════════════════
# FIG 1  fig_stablecoin_peg   —  2 panels STACKED vertically  (3.4 × 4.0)
# ══════════════════════════════════════════════════════════════════════════════
def fig_stablecoin_peg():
    ps = pd.Timestamp('2023-03-09', tz='UTC')
    pe = pd.Timestamp('2023-03-16', tz='UTC')
    pz = prices.loc[(prices.index >= ps) & (prices.index <= pe)]
    bz = basis.loc[(basis.index  >= ps) & (basis.index  <= pe)]

    with plt.rc_context(COL_RC):
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.4, 4.0), sharex=True)
        for ax in (ax1, ax2):
            shade_crisis(ax)
            ax.set_xlim(ps, pe)

        # Panel A – spot prices
        for col, lbl, c in [
            ('kraken_usdcusd',   'USDC/USD (Kraken)',   CB_GREEN),
            ('kraken_usdtusd',   'USDT/USD (Kraken)',   CB_BLUE),
            ('coinbase_usdtusd', 'USDT/USD (Coinbase)', CB_RED),
        ]:
            if col in pz.columns:
                ax1.plot(pz.index, pz[col], linewidth=0.65, alpha=0.9, label=lbl, color=c)
        ax1.axhline(1.0, color='grey', linewidth=0.7, linestyle='--', alpha=0.5)
        ax1.set_ylabel('Price (USD)')
        ax1.set_ylim(0.85, 1.04)
        ax1.set_title('(A) Stablecoin Spot Prices')
        ax1.legend(loc='lower right', frameon=True, framealpha=0.9,
                   fontsize=6, ncol=1)

        # Panel B – peg deviations
        ax2.axhline(0, color='black', linewidth=0.5, linestyle='--', alpha=0.4)
        for col, lbl, c in [
            ('usdc_peg_dev_kraken',   'USDC Dev (Kraken)',   CB_GREEN),
            ('usdt_peg_dev_kraken',   'USDT Dev (Kraken)',   CB_BLUE),
            ('usdt_peg_dev_coinbase', 'USDT Dev (Coinbase)', CB_RED),
        ]:
            if col in bz.columns:
                ax2.plot(bz.index, bz[col], linewidth=0.65, alpha=0.9, label=lbl, color=c)
        ax2.set_ylabel('Deviation (bps)')
        ax2.set_title('(B) Peg Deviations from $1.00')
        ax2.legend(loc='lower right', frameon=True, framealpha=0.9,
                   fontsize=6, ncol=1)
        fmt_date(ax2)

        fig.tight_layout(pad=0.5, h_pad=0.7)
        save(fig, 'fig_stablecoin_peg.png')


# ══════════════════════════════════════════════════════════════════════════════
# FIG 2  fig_dispersion_vs_adjusted_kraken  —  2 panels STACKED  (3.4 × 4.5)
# ══════════════════════════════════════════════════════════════════════════════
def fig_dispersion_vs_adjusted():
    zs = pd.Timestamp('2023-03-10', tz='UTC')
    ze = pd.Timestamp('2023-03-13 23:59', tz='UTC')
    bz = basis.loc[(basis.index >= zs) & (basis.index <= ze)]

    with plt.rc_context(COL_RC):
        fig, axes = plt.subplots(2, 1, figsize=(3.4, 4.5), sharex=True)
        for ax in axes:
            ax.axhline(0, color='black', linewidth=0.5, linestyle='--', alpha=0.4)
            shade_crisis(ax)

        for ax, d_col, b_col, title in [
            (axes[0], 'dispersion_usdc_kraken', 'basis_usdc_kraken',
             '(A) USDC Channel (Kraken)'),
            (axes[1], 'dispersion_usdt_kraken', 'basis_usdt_kraken',
             '(B) USDT Channel (Kraken)'),
        ]:
            if d_col in bz.columns:
                ax.plot(bz.index, bz[d_col], color=CB_ORANGE, linewidth=0.65,
                        alpha=0.9, label=r'$D_t$ (unadj.)')
            if b_col in bz.columns:
                ax.plot(bz.index, bz[b_col], color=CB_BLUE, linewidth=0.65,
                        alpha=0.9, label=r'$B_t$ (adj.)')
            ax.set_ylabel('bps')
            ax.set_title(title)
            ax.legend(loc='upper right', ncol=2, frameon=True,
                      framealpha=0.9, fontsize=6)

        fmt_date(axes[1], fmt='%b %d\n%H:%M', rotation=0)
        fig.tight_layout(pad=0.5, h_pad=0.7)
        save(fig, 'fig_dispersion_vs_adjusted_kraken.png')


# ══════════════════════════════════════════════════════════════════════════════
# FIG 3  fig_stablecoin_substitution_scatter  —  1 panel  (3.4 × 3.8)
# ══════════════════════════════════════════════════════════════════════════════
def fig_substitution_scatter():
    pre_mask    = (basis.index >= pd.Timestamp('2023-03-01', tz='UTC')) & \
                  (basis.index < svb_start)
    crisis_mask = (basis.index >= svb_start) & (basis.index < svb_end)

    scatter_df = basis.loc[
        pre_mask | crisis_mask,
        ['dispersion_usdc_kraken', 'dispersion_usdt_kraken'],
    ].copy().dropna()
    scatter_df.columns = ['D_USDC', 'D_USDT']
    scatter_df['Regime'] = np.where(
        (scatter_df.index >= svb_start) & (scatter_df.index < svb_end),
        'Crisis', 'Pre-SVB')

    r_pre    = scatter_df.loc[scatter_df['Regime']=='Pre-SVB',
               ['D_USDC','D_USDT']].corr().iloc[0,1]
    r_crisis = scatter_df.loc[scatter_df['Regime']=='Crisis',
               ['D_USDC','D_USDT']].corr().iloc[0,1]

    x_clip = scatter_df.loc[scatter_df['Regime']=='Crisis','D_USDC'].quantile(0.97)
    y_lo   = scatter_df.loc[scatter_df['Regime']=='Crisis','D_USDT'].quantile(0.01)
    plot_df = scatter_df.loc[
        (scatter_df['D_USDC'] >= -30) & (scatter_df['D_USDC'] <= x_clip) &
        (scatter_df['D_USDT'] >= y_lo) & (scatter_df['D_USDT'] <= 35)]

    with plt.rc_context(COL_RC):
        fig, ax = plt.subplots(figsize=(3.4, 3.8))
        pre_p = plot_df.loc[plot_df['Regime']=='Pre-SVB']
        cri_p = plot_df.loc[plot_df['Regime']=='Crisis']

        ax.scatter(pre_p['D_USDC'], pre_p['D_USDT'],
                   color=CB_GRAY, s=4, alpha=0.55, linewidths=0, zorder=2)
        ax.scatter(cri_p['D_USDC'], cri_p['D_USDT'],
                   color=CB_RED,  s=4, alpha=0.50, linewidths=0, zorder=3)
        for regime, colour in [('Pre-SVB','#555555'),('Crisis','#AA0000')]:
            sub = plot_df.loc[plot_df['Regime']==regime]
            ax.scatter(sub['D_USDC'].mean(), sub['D_USDT'].mean(),
                       color=colour, s=50, marker='D', zorder=5,
                       edgecolors='white', linewidths=0.7)

        ax.axhline(0, color='black', linewidth=0.4, linestyle='--', alpha=0.35)
        ax.axvline(0, color='black', linewidth=0.4, linestyle='--', alpha=0.35)

        annot = (r'$\rho$ (unadj. $D_t$):' '\n'
                 f'Pre-SVB: {r_pre:+.2f}\n'
                 f'Crisis:   {r_crisis:+.2f}\n\n'
                 r'$\rho$ (adj. $B_t$):' '\n'
                 r'Pre-SVB: $+$0.44' '\n'
                 r'Crisis:   $+$0.16')
        ax.text(0.97, 0.97, annot, transform=ax.transAxes,
                fontsize=6, va='top', ha='right', linespacing=1.4,
                bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                          edgecolor='#CCCCCC', linewidth=0.6, alpha=0.95))

        legend_els = [
            Line2D([0],[0],marker='o',color='w',markerfacecolor=CB_GRAY,
                   markersize=5,label=f'Pre-SVB ($n={len(pre_p):,}$)',alpha=0.8),
            Line2D([0],[0],marker='o',color='w',markerfacecolor=CB_RED,
                   markersize=5,label=f'Crisis ($n={len(cri_p):,}$)',alpha=0.8),
            Line2D([0],[0],marker='D',color='w',markerfacecolor='#555555',
                   markeredgecolor='white',markeredgewidth=0.5,markersize=5,
                   label='Pre-SVB centroid'),
            Line2D([0],[0],marker='D',color='w',markerfacecolor='#AA0000',
                   markeredgecolor='white',markeredgewidth=0.5,markersize=5,
                   label='Crisis centroid'),
        ]
        ax.legend(handles=legend_els, loc='upper left',
                  frameon=True, framealpha=0.9, edgecolor='#CCCCCC', fontsize=6)
        ax.set_xlabel(r'$D_{USDC,t}$ (bps)')
        ax.set_ylabel(r'$D_{USDT,t}$ (bps)')
        ax.set_title(r'Stablecoin Substitution ($D_t$, Kraken)')
        ax.yaxis.grid(True, linewidth=0.3, color='#EEEEEE')
        ax.set_axisbelow(True)
        fig.tight_layout(pad=0.4)
        save(fig, 'fig_stablecoin_substitution_scatter.png')


# ══════════════════════════════════════════════════════════════════════════════
# FIG 4  fig_half_life_robustness  —  3 regime cols side-by-side  (3.4 × 2.6)
# (kept horizontal: dot-plot cells are compact enough at ~1.1" each)
# ══════════════════════════════════════════════════════════════════════════════
def fig_half_life_robustness():
    csv_path = os.path.join(TABLES_DIR, 'half_life_robustness.csv')
    if not os.path.exists(csv_path):
        print("  SKIP fig_half_life_robustness (CSV not found)")
        return
    df = pd.read_csv(csv_path)
    df['spec']       = df['freq'] + ' ' + df['ff_filter']
    df['stablecoin'] = df['series'].str.extract(r'(USDC|USDT)')[0]

    regimes = ['Pre-SVB', 'Crisis', 'Post-SVB']
    specs   = ['1m all', '1m no_ff', '5m all', '5m no_ff']
    colors  = {'USDC': CB_BLUE, 'USDT': CB_RED}
    markers = {'USDC': 'o', 'USDT': 's'}

    with plt.rc_context(COL_RC):
        fig, axes = plt.subplots(1, 3, figsize=(3.4, 2.6), sharey=True)
        for ax, regime in zip(axes, regimes):
            sub = df[df['regime'] == regime]
            for j, coin in enumerate(['USDC', 'USDT']):
                csub = sub[sub['stablecoin'] == coin]
                for i, spec in enumerate(specs):
                    row = csub[csub['spec'] == spec]
                    x   = i + (j - 0.5) * 0.20
                    if len(row) == 0 or pd.isna(row['half_life_min'].values[0]):
                        ax.scatter(x, 0, marker='x', color=colors[coin],
                                   s=20, zorder=5)
                        ax.annotate('NaN', (x, 0.1), fontsize=5, ha='center',
                                    color=colors[coin])
                    else:
                        val = row['half_life_min'].values[0]
                        ax.scatter(x, val, marker=markers[coin],
                                   color=colors[coin], s=22, zorder=5,
                                   edgecolors='k', linewidths=0.3)
                        ax.annotate(f'{val:.1f}', (x, val+0.09),
                                    fontsize=5, ha='center', color=colors[coin])
            ax.set_title(regime, fontsize=7, fontweight='bold')
            ax.set_xticks(range(len(specs)))
            ax.set_xticklabels(specs, fontsize=5.5, rotation=35, ha='right')
            ax.set_xlim(-0.5, len(specs)-0.5)
            ax.grid(axis='y', alpha=0.3, linewidth=0.3)

        axes[0].set_ylabel('Half-Life (min)', fontsize=7)
        legend_els = [
            Line2D([0],[0],marker='o',color='w',markerfacecolor=CB_BLUE,
                   markeredgecolor='k',markersize=5,label='USDC $B_t$'),
            Line2D([0],[0],marker='s',color='w',markerfacecolor=CB_RED,
                   markeredgecolor='k',markersize=5,label='USDT $B_t$'),
        ]
        fig.legend(handles=legend_els, loc='lower center', ncol=2,
                   fontsize=6, frameon=False, bbox_to_anchor=(0.5, -0.06))
        fig.suptitle('Half-Life Robustness: Adjusted Residual $B_t$',
                     fontsize=8, fontweight='bold', y=1.02)
        fig.tight_layout(pad=0.4, w_pad=0.3)
        save(fig, 'fig_half_life_robustness.png')


# ══════════════════════════════════════════════════════════════════════════════
# FIG 5  fig_two_layer_persistence  —  1 panel dual-axis  (3.4 × 3.0)
# ══════════════════════════════════════════════════════════════════════════════
def fig_two_layer_persistence():
    t0 = pd.Timestamp('2023-03-10', tz='UTC')
    t1 = pd.Timestamp('2023-03-13', tz='UTC')
    d  = basis.loc[(basis.index >= t0) & (basis.index < t1)]

    with plt.rc_context(COL_RC):
        fig, ax1 = plt.subplots(figsize=(3.4, 3.0))

        line_bt, = ax1.plot(d.index, d['basis_usdc_kraken'],
                            color=CB_BLUE, linewidth=0.55, alpha=0.90, zorder=3,
                            label=r'$B_{USDC,t}$ (Kraken, left)')
        ax1.axhline(0, color='black', linewidth=0.4, linestyle='--', alpha=0.35)
        ax1.set_ylabel(r'$B_{USDC,t}$ (bps)', color=CB_BLUE, labelpad=3)
        ax1.tick_params(axis='y', labelcolor=CB_BLUE)
        ax1.set_ylim(-130, 210)
        ax1.yaxis.set_major_locator(mticker.MultipleLocator(50))

        ax2 = ax1.twinx()
        ax2.spines['right'].set_visible(True)
        ax2.spines['right'].set_linewidth(0.6)
        line_peg, = ax2.plot(d.index, d['usdc_peg_dev_kraken'],
                             color=CB_RED, linewidth=0.9, alpha=0.92, zorder=2,
                             label='Peg Dev. (right)')
        ax2.set_ylabel('USDC Peg Dev. (bps)', color=CB_RED, labelpad=3)
        ax2.tick_params(axis='y', labelcolor=CB_RED)
        ax2.set_ylim(-1450, 230)
        ax2.yaxis.set_major_locator(mticker.MultipleLocator(250))

        ax1.xaxis.set_major_locator(mdates.HourLocator(interval=12))
        ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:%M'))
        plt.setp(ax1.xaxis.get_majorticklabels(), ha='center', fontsize=6)
        ax1.set_xlim(t0, t1)
        ax1.set_xlabel('Date / Time (UTC)', labelpad=3)

        lines  = [line_bt, line_peg]
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc='upper left', frameon=True,
                   framealpha=0.92, edgecolor='#CCCCCC', fontsize=6)

        ax1.text(0.995, 0.97,
            r'HL gap: $\times$940' '\n'
            r'$B_{USDC,t}$: $\approx$0.6 min' '\n'
            r'Peg dev: $\approx$572 min',
            transform=ax1.transAxes, fontsize=6,
            va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#F7F7F7',
                      edgecolor='#CCCCCC', linewidth=0.6, alpha=0.95))

        ax1.set_title(
            r'Two-Layer Persistence: $B_{USDC,t}$ vs.\ Peg Dev.' '\n'
            '(Kraken, 10--13 Mar 2023)', pad=4)
        fig.tight_layout(pad=0.5)
        save(fig, 'fig_two_layer_persistence.png')


# ══════════════════════════════════════════════════════════════════════════════
# FIG 6  fig_cross_exchange_basis  —  3 panels STACKED vertically  (3.4 × 5.5)
# ══════════════════════════════════════════════════════════════════════════════
def fig_cross_exchange_basis():
    zs = pd.Timestamp('2023-03-09', tz='UTC')
    ze = pd.Timestamp('2023-03-21 23:59', tz='UTC')
    bz = basis.loc[(basis.index >= zs) & (basis.index <= ze)]

    specs = [
        ('xbasis_btcusdt_binance_kraken',  '(A) BTC/USDT Binance−Kraken', CB_ORANGE),
        ('xbasis_btcusdt_coinbase_kraken',  '(B) BTC/USDT Coinbase−Kraken', CB_CYAN),
        ('xbasis_btcusd_coinbase_kraken',   '(C) BTC/USD Coinbase−Kraken',  CB_PURPLE),
    ]

    with plt.rc_context(COL_RC):
        fig, axes = plt.subplots(3, 1, figsize=(3.4, 5.5), sharex=True)
        for ax, (col, title, c) in zip(axes, specs):
            shade_crisis(ax)
            ax.axhline(0, color='black', linewidth=0.4, linestyle='--', alpha=0.4)
            ax.set_xlim(zs, ze)
            if col in bz.columns:
                ax.plot(bz.index, bz[col], linewidth=0.5, color=c, alpha=0.85)
            ax.set_title(title, fontsize=7.5)
            ax.set_ylabel('Basis (bps)')

        fmt_date(axes[2])
        fig.tight_layout(pad=0.5, h_pad=0.6)
        save(fig, 'fig_cross_exchange_basis.png')


# ══════════════════════════════════════════════════════════════════════════════
# FIG 7  fig_liquidity_roll_amihud  —  2 panels STACKED vertically  (3.4 × 4.5)
# ══════════════════════════════════════════════════════════════════════════════
def fig_liquidity_roll_amihud():
    REGIME_ORDER = ['Pre-SVB', 'Crisis', 'Post-SVB']
    PAIRS = {
        'Kraken BTC/USD':   ('kraken_btcusd',   'kraken_btcusd'),
        'Kraken BTC/USDT':  ('kraken_btcusdt',  'kraken_btcusdt'),
        'Kraken BTC/USDC':  ('kraken_btcusdc',  'kraken_btcusdc'),
        'Binance BTC/USDT': ('binance_btcusdt', 'binance_btcusdt'),
        'Coinbase BTC/USD': ('coinbase_btcusd', 'coinbase_btcusd'),
    }
    PAIR_ORDER = list(PAIRS.keys())
    short_names = ['KR/USD', 'KR/USDT', 'KR/USDC', 'BN/USDT', 'CB/USD']

    def roll_spread_daily(price_col):
        p  = prices[price_col].dropna()
        lr = np.log(p / p.shift(1))
        rows = []
        for date, grp in lr.groupby(lr.index.date):
            r = grp.dropna().values
            if len(r) < 15: continue
            cov = np.cov(r[1:], r[:-1])[0, 1]
            rows.append({'date': pd.Timestamp(date),
                         'roll_bps': 2.0*np.sqrt(-cov)*10000 if cov < 0 else np.nan})
        if not rows: return pd.Series(dtype=float)
        s = pd.DataFrame(rows).set_index('date')['roll_bps']
        s.index = pd.DatetimeIndex(s.index).tz_localize('UTC')
        return s

    def amihud_daily(price_col, vol_col):
        if vol_col not in volumes.columns: return pd.Series(dtype=float)
        p    = prices[price_col]; v = volumes[vol_col]
        lr   = np.log(p / p.shift(1)).abs()
        dvol = v * p
        aligned = pd.concat([lr, dvol], axis=1, keys=['abs_ret','dvol']).dropna()
        aligned  = aligned[aligned['dvol'] > 1.0]
        aligned['illiq'] = aligned['abs_ret'] / aligned['dvol']
        daily = aligned.groupby(aligned.index.date)['illiq'].mean() * 1e6
        s = daily.copy(); s.index = pd.DatetimeIndex(s.index).tz_localize('UTC')
        return s

    roll_series   = {lbl: roll_spread_daily(pc) for lbl, (pc, _)  in PAIRS.items()
                     if pc in prices.columns}
    amihud_series = {lbl: amihud_daily(pc, vc) for lbl, (pc, vc) in PAIRS.items()
                     if pc in prices.columns}

    def regime_stats(daily_dict):
        rows = []
        for lbl in PAIR_ORDER:
            if lbl not in daily_dict: continue
            s = daily_dict[lbl]
            for reg in REGIME_ORDER:
                if reg == 'Pre-SVB':  mask = s.index < svb_start
                elif reg == 'Crisis': mask = (s.index >= svb_start) & (s.index < svb_end)
                else:                 mask = s.index >= svb_end
                sub = s[mask].dropna()
                rows.append({'Pair': lbl, 'Regime': reg,
                             'mean': round(sub.mean(), 3) if len(sub) else np.nan,
                             'N':    len(sub)})
        return pd.DataFrame(rows)

    df_roll   = regime_stats(roll_series)
    df_amihud = regime_stats(amihud_series)

    reg_colors = {'Pre-SVB': CB_BLUE, 'Crisis': CB_RED, 'Post-SVB': CB_GREEN}
    w = 0.22

    def grouped_bar(ax, df, ylabel, title):
        pivot = df.pivot(index='Pair', columns='Regime', values='mean')[REGIME_ORDER]
        pivot = pivot.reindex(PAIR_ORDER)
        x = np.arange(len(pivot))
        for i, reg in enumerate(REGIME_ORDER):
            ax.bar(x + (i-1)*w, pivot[reg], w,
                   label=reg, color=reg_colors[reg], alpha=0.85,
                   edgecolor='white', linewidth=0.3)
        ax.set_xticks(x)
        ax.set_xticklabels(short_names, fontsize=6)
        ax.set_ylabel(ylabel, fontsize=7)
        ax.set_title(title, fontsize=7.5)
        ax.legend(fontsize=6, ncol=3)
        ax.grid(axis='y', linewidth=0.3, alpha=0.5)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)

    with plt.rc_context(COL_RC):
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.4, 4.5))
        grouped_bar(ax1, df_roll,
                    'Roll Spread (bps)',
                    '(A) Roll (1984) Effective Spread')
        grouped_bar(ax2, df_amihud,
                    r'Amihud ILLIQ ($\times10^{-6}$)',
                    '(B) Amihud (2002) ILLIQ Ratio')
        fig.tight_layout(pad=0.5, h_pad=0.8)
        save(fig, 'fig_liquidity_roll_amihud.png')


# ══════════════════════════════════════════════════════════════════════════════
# FIG 8  fig_volume_share  —  1 panel  (3.4 × 3.0)
# ══════════════════════════════════════════════════════════════════════════════
def fig_volume_share():
    vol_cols  = ['binance_btcusdt', 'binance_btcusdc', 'coinbase_btcusd',
                 'coinbase_btcusdt', 'kraken_btcusd', 'kraken_btcusdt', 'kraken_btcusdc']
    available = [c for c in vol_cols if c in volumes.columns]
    if not available:
        print("  SKIP fig_volume_share (no volume columns found)"); return
    vols_daily = volumes[available].resample('D').sum()
    vols_pct   = vols_daily.div(vols_daily.sum(axis=1), axis=0) * 100

    label_map = {
        'binance_btcusdt': 'Binance USDT',  'binance_btcusdc': 'Binance USDC',
        'coinbase_btcusd': 'Coinbase USD',  'coinbase_btcusdt': 'Coinbase USDT',
        'kraken_btcusd':   'Kraken USD',    'kraken_btcusdt':   'Kraken USDT',
        'kraken_btcusdc':  'Kraken USDC',
    }
    colors_vol = [CB_BLUE, CB_CYAN, CB_ORANGE, CB_PURPLE, '#444444', CB_GREEN, CB_RED]

    with plt.rc_context(COL_RC):
        fig, ax = plt.subplots(figsize=(3.4, 3.0))
        ax.stackplot(
            vols_pct.index,
            *[vols_pct[c] for c in available],
            labels=[label_map.get(c, c) for c in available],
            colors=colors_vol[:len(available)],
            alpha=0.82,
        )
        ax.axvspan(svb_start.normalize(), svb_end.normalize(),
                   alpha=0.25, color='red', label='SVB Crisis', zorder=10)
        ax.set_title('Daily Volume Fragmentation')
        ax.set_ylabel('Volume Share (%)')
        fmt_date(ax)
        ax.legend(loc='upper left', fontsize=5.5, frameon=True,
                  framealpha=0.9, ncol=2)
        fig.tight_layout(pad=0.4)
        save(fig, 'fig_volume_share.png')


# ══════════════════════════════════════════════════════════════════════════════
# FIG 9  fig_realized_volatility  —  1 panel  (3.4 × 3.0)
# ══════════════════════════════════════════════════════════════════════════════
def fig_realized_volatility():
    vol_cols_btc = ['kraken_btcusd', 'kraken_btcusdt', 'kraken_btcusdc',
                    'binance_btcusdt', 'coinbase_btcusd']
    available = [c for c in vol_cols_btc if c in prices.columns]
    rv = prices[available].pct_change(fill_method=None).rolling(60).std() * 10000 * np.sqrt(60)

    nice_names = {
        'kraken_btcusd':  'Kraken USD',   'kraken_btcusdt': 'Kraken USDT',
        'kraken_btcusdc': 'Kraken USDC',  'binance_btcusdt':'Binance USDT',
        'coinbase_btcusd':'Coinbase USD',
    }
    colors_rv = ['#2c3e50', CB_BLUE, CB_GREEN, CB_ORANGE, CB_RED]

    with plt.rc_context(COL_RC):
        fig, ax = plt.subplots(figsize=(3.4, 3.0))
        shade_crisis(ax)
        for col, c in zip(available, colors_rv):
            ax.plot(rv.index, rv[col], linewidth=0.5, color=c,
                    label=nice_names.get(col, col), alpha=0.85)
        ax.set_title('Hourly Realized Volatility (60-min, $\\times\\sqrt{60}$)')
        ax.set_ylabel('Volatility (bps/hr)')
        fmt_date(ax)
        ax.legend(loc='upper right', fontsize=6, ncol=1, frameon=True)
        fig.tight_layout(pad=0.4)
        save(fig, 'fig_realized_volatility.png')


# ══════════════════════════════════════════════════════════════════════════════
# FIG 10  fig_tail_blowout_kde  —  1 panel  (3.4 × 3.2)
# ══════════════════════════════════════════════════════════════════════════════
def fig_tail_blowout_kde():
    pre_mask    = (basis.index >= pd.Timestamp('2023-03-01', tz='UTC')) & \
                  (basis.index < svb_start)
    crisis_mask = (basis.index >= svb_start) & (basis.index < svb_end)
    pre_bt    = basis.loc[pre_mask,    'basis_usdc_kraken'].dropna()
    crisis_bt = basis.loc[crisis_mask, 'basis_usdc_kraken'].dropna()
    x_lo, x_hi = -50, 100

    with plt.rc_context(COL_RC):
        fig, ax = plt.subplots(figsize=(3.4, 3.2))

        sns.kdeplot(pre_bt,    ax=ax, color=CB_GRAY, linewidth=1.4,
                    fill=True, alpha=0.40,
                    label=f'Pre-SVB ($n={len(pre_bt):,}$)',
                    clip=(x_lo, x_hi), bw_adjust=0.9)
        sns.kdeplot(crisis_bt, ax=ax, color=CB_RED,  linewidth=1.4,
                    fill=True, alpha=0.35,
                    label=f'Crisis ($n={len(crisis_bt):,}$)',
                    clip=(x_lo, x_hi), bw_adjust=0.9)

        ax.axvline(0,               color='black',  linewidth=0.5, linestyle='--', alpha=0.45)
        ax.axvline(pre_bt.mean(),   color=CB_GRAY,  linewidth=1.0, linestyle=':',
                   label=f'Pre-SVB mean ({pre_bt.mean():.1f} bps)')
        ax.axvline(crisis_bt.mean(),color=CB_RED,   linewidth=1.0, linestyle=':',
                   label=f'Crisis mean ({crisis_bt.mean():.1f} bps)')

        p99_pre, p99_crisis = pre_bt.quantile(0.99), crisis_bt.quantile(0.99)
        ax.text(0.97, 0.97,
            f'P99 expansion\nPre-SVB: {p99_pre:.1f} bps\nCrisis: {p99_crisis:.1f} bps\nRatio: \u00d76',
            transform=ax.transAxes, fontsize=6,
            va='top', ha='right', linespacing=1.4,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF5F5',
                      edgecolor=CB_RED, linewidth=0.7, alpha=0.96))

        ax.text(0.03, 0.97,
            f'Ex. kurtosis\nPre-SVB: {pre_bt.kurt():.1f}\nCrisis: {crisis_bt.kurt():.1f}',
            transform=ax.transAxes, fontsize=6,
            va='top', ha='left', linespacing=1.4,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#F5F5FF',
                      edgecolor=CB_BLUE, linewidth=0.7, alpha=0.96))

        ax.set_xlim(x_lo, x_hi)
        ax.set_xlabel(r'$B_{USDC,t}$ (bps)')
        ax.set_ylabel('Density')
        ax.yaxis.grid(True, linewidth=0.3, color='#EEEEEE')
        ax.set_axisbelow(True)
        ax.legend(loc='upper center', frameon=True, framealpha=0.9,
                  edgecolor='#CCCCCC', fontsize=6)
        ax.set_title(r'Tail Blowout: $B_{USDC,t}$ (Kraken)', pad=4)
        fig.tight_layout(pad=0.4)
        save(fig, 'fig_tail_blowout_kde.png')


# ══════════════════════════════════════════════════════════════════════════════
# FIG 11  fig_correlation_regime_heatmap  —  2 on top (Pre-SVB, Crisis) + 1 centred
# below (Post-SVB), triangle layout.  Each heatmap is forced square via aspect='equal'.
# ══════════════════════════════════════════════════════════════════════════════
def fig_correlation_regime_heatmap():
    pre_mask    = (basis.index >= pd.Timestamp('2023-03-01', tz='UTC')) & \
                  (basis.index < svb_start)
    crisis_mask = (basis.index >= svb_start) & (basis.index < svb_end)
    post_mask   = (basis.index >= pd.Timestamp('2023-03-13', tz='UTC')) & \
                  (basis.index < pd.Timestamp('2023-03-22', tz='UTC'))

    corr_cols = [
        'basis_usdc_kraken', 'basis_usdt_kraken', 'basis_usdt_coinbase',
        'xbasis_btcusdt_binance_kraken', 'xbasis_btcusd_coinbase_kraken',
    ]
    corr_labels = [
        r'$B_{USDC}$ KR', r'$B_{USDT}$ KR', r'$B_{USDT}$ CB',
        r'X-USDT BN-KR',  r'X-USD CB-KR',
    ]

    def _draw_heatmap(ax, mask, regime_name):
        sub = basis.loc[mask, corr_cols].dropna()
        corr_mat = sub.corr()
        im = ax.imshow(corr_mat.values, vmin=-0.2, vmax=1.0,
                       cmap='RdBu_r', aspect='equal')
        n = len(corr_cols)
        for i in range(n):
            for j in range(n):
                val   = corr_mat.values[i, j]
                color = 'white' if abs(val) > 0.55 else 'black'
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=5.5, color=color, fontweight='bold')
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xticklabels(corr_labels, rotation=45, ha='right', fontsize=5.5)
        ax.set_yticklabels(corr_labels, fontsize=5.5)
        ax.set_title(f'{regime_name}  ($n={len(sub):,}$)', fontsize=7, pad=3)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(length=0)
        return im

    with plt.rc_context(COL_RC):
        # 2×2 grid; top-left=Pre-SVB, top-right=Crisis, bottom-centre=Post-SVB.
        # aspect='auto' on imshow, but we force equal axes via set_box_aspect(1)
        # so every heatmap is physically square and identical in size.
        fig, axes = plt.subplots(2, 2, figsize=(6.8, 6.0),
                                 gridspec_kw={'hspace': 0.65, 'wspace': 0.45})

        ax_pre    = axes[0, 0]
        ax_crisis = axes[0, 1]
        # Hide top-right placeholder; Post-SVB goes in the centred bottom slot
        axes[1, 0].set_visible(False)
        axes[1, 1].set_visible(False)

        # Add a new axes manually centred in the bottom row
        # GridSpec bottom-row spans cols 0–1, so centre = midpoint of the two col slots
        gs = axes[1, 0].get_gridspec()
        ax_post = fig.add_subplot(gs[1, 0:2])
        # Shrink it to the same width as one column slot and centre it
        pos0 = axes[0, 0].get_position()   # left panel position
        pos1 = axes[0, 1].get_position()   # right panel position
        panel_w = pos0.width
        panel_h = pos0.height
        centre_x = (pos0.x0 + pos1.x1) / 2.0
        pos_bot = axes[1, 0].get_position()
        ax_post.set_position([centre_x - panel_w / 2.0,
                               pos_bot.y0,
                               panel_w,
                               panel_h])

        im = _draw_heatmap(ax_pre,    pre_mask,    'Pre-SVB')
        _draw_heatmap(ax_crisis, crisis_mask, 'Crisis')
        _draw_heatmap(ax_post,   post_mask,   'Post-SVB')

        # Force all three to have identical square aspect
        for ax in (ax_pre, ax_crisis, ax_post):
            ax.set_box_aspect(1)

        # Colorbar anchored to the right of the top-right panel
        cbar_ax = fig.add_axes([pos1.x1 + 0.02, pos1.y0, 0.02, pos1.height])
        cbar = fig.colorbar(im, cax=cbar_ax)
        cbar.set_label('Pearson Corr.', fontsize=6.5)
        cbar.ax.tick_params(labelsize=6)

        fig.suptitle('Cross-Channel Correlation by Regime',
                     fontsize=8, y=0.99)
        fig.savefig(os.path.join(FIGURES_COL, 'fig_correlation_regime_heatmap.png'),
                    dpi=300, bbox_inches='tight')
        plt.close(fig)
        print("  saved → fig_correlation_regime_heatmap.png")


# ══════════════════════════════════════════════════════════════════════════════
# FIG 12  fig_arbitrage_after_fees  —  2 panels STACKED vertically  (3.4 × 4.5)
# ══════════════════════════════════════════════════════════════════════════════
def fig_arbitrage_after_fees():
    FEE_BPS = 5.0
    arb_specs = [
        {'key':    'basis_usdc_kraken',
         'basis':  'basis_usdc_kraken',
         'label':  'USDC/USD KR (3-leg)',
         'n_legs': 3,
         'ranges': ['kraken_btcusdc', 'kraken_usdcusd', 'kraken_btcusd']},
        {'key':    'basis_usdt_kraken',
         'basis':  'basis_usdt_kraken',
         'label':  'USDT/USD KR (3-leg)',
         'n_legs': 3,
         'ranges': ['kraken_btcusdt', 'kraken_usdtusd', 'kraken_btcusd']},
        {'key':    'xbasis_btcusdt_binance_kraken',
         'basis':  'xbasis_btcusdt_binance_kraken',
         'label':  'USDT BN-KR (2-leg)',
         'n_legs': 2,
         'ranges': ['binance_btcusdt', 'kraken_btcusdt']},
        {'key':    'xbasis_btcusd_coinbase_kraken',
         'basis':  'xbasis_btcusd_coinbase_kraken',
         'label':  'USD CB-KR (2-leg)',
         'n_legs': 2,
         'ranges': ['coinbase_btcusd', 'kraken_btcusd']},
    ]
    colors_arb = [CB_ORANGE, CB_BLUE, CB_RED, '#2c3e50']
    chan_data  = {}
    for spec in arb_specs:
        if spec['basis'] not in basis.columns: continue
        if any(r not in ranges.columns for r in spec['ranges']): continue
        df = pd.DataFrame(index=basis.index)
        df['abs_basis'] = basis[spec['basis']].abs()
        for i, rc in enumerate(spec['ranges'], 1):
            df[f'lr{i}'] = ranges[rc] * 10000.0
        df = df.dropna()
        if df.empty: continue
        df['fee']     = spec['n_legs'] * FEE_BPS
        df['slip']    = 0.5 * df[[f'lr{i}' for i in range(1,len(spec['ranges'])+1)]].sum(axis=1)
        df['net_fee'] = (df['abs_basis'] - df['fee']).clip(lower=0.0)
        df['net_fs']  = (df['abs_basis'] - df['fee'] - df['slip']).clip(lower=0.0)
        chan_data[spec['key']] = (spec['label'], df)

    if not chan_data:
        print("  SKIP fig_arbitrage_after_fees (no channels available)"); return

    with plt.rc_context(COL_RC):
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.4, 4.5), sharex=True)
        for ax in (ax1, ax2):
            shade_crisis(ax)

        for (lbl, df), c in zip(chan_data.values(), colors_arb):
            ax1.plot(df.index, df['net_fee'], linewidth=0.45, color=c,
                     label=lbl, alpha=0.85)
            ax2.plot(df.index, df['net_fs'],  linewidth=0.45, color=c,
                     label=lbl, alpha=0.85)

        ax1.set_title(f'(A) Fee-Only Net Arb. ({FEE_BPS:.0f} bps/leg)')
        ax1.set_ylabel('Net Profit (bps)')
        ax1.legend(loc='upper left', fontsize=6, ncol=1, frameon=True)

        ax2.set_title('(B) Fee + Slippage Net Arb.')
        ax2.set_ylabel('Net Profit (bps)')
        fmt_date(ax2)

        fig.tight_layout(pad=0.5, h_pad=0.7)
        save(fig, 'fig_arbitrage_after_fees.png')


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
if __name__ == '__main__':
    os.chdir(ROOT)
    print(f"\nGenerating single-column figures → {FIGURES_COL}\n")

    fig_stablecoin_peg()
    fig_dispersion_vs_adjusted()
    fig_substitution_scatter()
    fig_half_life_robustness()
    fig_two_layer_persistence()
    fig_cross_exchange_basis()
    fig_liquidity_roll_amihud()
    fig_volume_share()
    fig_realized_volatility()
    fig_tail_blowout_kde()
    fig_correlation_regime_heatmap()
    fig_arbitrage_after_fees()

    print(f"\nAll figures written to {FIGURES_COL}/")



Generating single-column figures → /Users/dhruvkohli/Desktop/Github Repos/IAQF2026/figures_col

  saved → fig_stablecoin_peg.png
  saved → fig_dispersion_vs_adjusted_kraken.png
  saved → fig_stablecoin_substitution_scatter.png
  saved → fig_half_life_robustness.png
  saved → fig_two_layer_persistence.png
  saved → fig_cross_exchange_basis.png
  saved → fig_liquidity_roll_amihud.png
  saved → fig_volume_share.png
  saved → fig_realized_volatility.png
  saved → fig_tail_blowout_kde.png
  saved → fig_correlation_regime_heatmap.png
  saved → fig_arbitrage_after_fees.png

All figures written to /Users/dhruvkohli/Desktop/Github Repos/IAQF2026/figures_col/


---
# Section 9: Additional Figures — SVB Crisis Zoom & VAR IRF

From `src/09_additional_figures.py`:
- **Figure 6**: SVB Crisis Zoom (all channels, March 10–13)
- **Figure 13**: VAR Impulse-Response Function (BTC/USD vs BTC/USDC on Kraken)


In [23]:
# Additional figures (SVB zoom + VAR IRF)
# Data and returns already loaded above

# ── Colour palette (Paul Tol "bright" – colour-blind safe) ────────────────────
CB_BLUE   = '#4477AA'
CB_RED    = '#EE6677'
CB_GREEN  = '#228833'
CB_ORANGE = '#CCBB44'
CB_PURPLE = '#AA3377'
CB_CYAN   = '#66CCEE'
CB_DARK   = '#332288'

# ── Full-width RC (for figures/) ──────────────────────────────────────────────
FULL_RC = {
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset':  'stix',
    'axes.facecolor':    'white',
    'figure.facecolor':  'white',
    'axes.grid':         False,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.linewidth':    0.8,
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.labelsize':    12,
    'legend.fontsize':   10,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'xtick.direction':   'out',
    'ytick.direction':   'out',
}

# ── Single-column RC (for figures_col/) ───────────────────────────────────────
COL_RC = {
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset':  'stix',
    'axes.facecolor':    'white',
    'figure.facecolor':  'white',
    'axes.grid':         False,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.linewidth':    0.6,
    'font.size':         7,
    'axes.titlesize':    8,
    'axes.labelsize':    7.5,
    'legend.fontsize':   6.5,
    'xtick.labelsize':   6.5,
    'ytick.labelsize':   6.5,
    'xtick.direction':   'out',
    'ytick.direction':   'out',
    'lines.linewidth':   0.65,
    'xtick.major.size':  2.5,
    'ytick.major.size':  2.5,
}


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 – SVB Crisis Zoom (all channels, March 10-13)
# ══════════════════════════════════════════════════════════════════════════════
def plot_crisis_zoom(rc_dict, figsize_a, figsize_b, lw, save_dir, dpi=300):
    """
    Two-panel figure:
      Panel A: Intra-exchange adjusted residual B_t for all three channels
      Panel B: Cross-exchange basis for BTC/USDT (Binance-Kraken) and BTC/USD (Coinbase-Kraken)
    """
    svb_data = basis.loc[(basis.index >= svb_start) & (basis.index < svb_end)]

    with plt.rc_context(rc_dict):
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(figsize_a, figsize_b),
                                        sharex=True)

        # --- Panel A: Intra-exchange B_t ---
        ax1.axhline(0, color='black', linewidth=0.6, linestyle='--', alpha=0.35)
        for col, lbl, c in [
            ('basis_usdc_kraken',   r'$B_{USDC,t}$ (Kraken)',   CB_GREEN),
            ('basis_usdt_kraken',   r'$B_{USDT,t}$ (Kraken)',   CB_BLUE),
            ('basis_usdt_coinbase', r'$B_{USDT,t}$ (Coinbase)', CB_RED),
        ]:
            if col in svb_data.columns:
                ax1.plot(svb_data.index, svb_data[col],
                         linewidth=lw, alpha=0.88, label=lbl, color=c)
        ax1.set_ylabel(r'Adjusted Residual $B_t$ (bps)')
        ax1.set_title('(A) Intra-Exchange Adjusted Residuals')
        ax1.legend(loc='upper right', frameon=True, framealpha=0.92,
                   edgecolor='#CCCCCC')

        # --- Panel B: Cross-exchange basis ---
        ax2.axhline(0, color='black', linewidth=0.6, linestyle='--', alpha=0.35)
        for col, lbl, c in [
            ('xbasis_btcusdt_binance_kraken', r'Binance$-$Kraken BTC/USDT', CB_ORANGE),
            ('xbasis_btcusd_coinbase_kraken', r'Coinbase$-$Kraken BTC/USD', CB_DARK),
        ]:
            if col in svb_data.columns:
                ax2.plot(svb_data.index, svb_data[col],
                         linewidth=lw, alpha=0.88, label=lbl, color=c)
        ax2.set_ylabel('Cross-Exchange Basis (bps)')
        ax2.set_title('(B) Cross-Exchange Basis')
        ax2.legend(loc='upper right', frameon=True, framealpha=0.92,
                   edgecolor='#CCCCCC')

        # --- Shared x-axis ---
        ax2.xaxis.set_major_locator(mdates.HourLocator(interval=12))
        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:%M'))
        plt.setp(ax2.xaxis.get_majorticklabels(), ha='center')
        ax2.set_xlim(svb_start, svb_end)

        fig.tight_layout(pad=0.5, h_pad=0.8)
        path = os.path.join(save_dir, 'fig_svb_crisis_zoom.png')
        fig.savefig(path, dpi=dpi, bbox_inches='tight')
        plt.close(fig)
        print(f"  saved → {path}")


# Full-width version
plot_crisis_zoom(FULL_RC, figsize_a=12, figsize_b=7, lw=0.55,
                 save_dir=FIGURES_DIR)
# Column version
plot_crisis_zoom(COL_RC, figsize_a=3.4, figsize_b=4.5, lw=0.45,
                 save_dir=FIGURES_COL)


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 – VAR Impulse-Response Function (BTC/USD vs BTC/USDC, Kraken)
#
# Methodology notes:
#   - VAR on 1-minute percentage returns (×10000 → bps) for Kraken BTC/USD
#     and BTC/USDC.  Lag selected by AIC (maxlags=10).
#   - Orthogonalized IRF via Cholesky decomposition.
#   - Ordering: [BTC/USD, BTC/USDC]. BTC/USD is ordered first because it is
#     more liquid and the identified price-discovery leader (Hasbrouck IS = 0.73).
#     Under this ordering, BTC/USD can affect BTC/USDC contemporaneously but
#     not vice versa.
#   - IRF horizon: 10 lags (= 10 minutes at 1-min frequency).
#   - 95% confidence bands from bootstrap (default in statsmodels).
#   - Full-sample estimation (March 1-21, 2023); reflects average dynamics
#     across all three regimes.
# ══════════════════════════════════════════════════════════════════════════════

# --- Fit VAR model ---
var_data = returns[['kraken_btcusd', 'kraken_btcusdc']].dropna() * 10000
var_model = VAR(var_data)
var_result = var_model.fit(maxlags=10, ic='aic')
selected_lags = var_result.k_ar
n_obs = len(var_data)
print(f"\nVAR: AIC-selected lags = {selected_lags}, N = {n_obs}")

# --- Compute orthogonalized IRF with asymptotic 95% CI ---
irf = var_result.irf(10)

# Extract arrays: shape (steps+1, n_vars, n_vars) where steps=10 → 11 points
irf_vals  = irf.orth_irfs                         # (11, 2, 2)
irf_se    = irf.stderr(orth=True)                  # (11, 2, 2) asymptotic SE
irf_lower = irf_vals - 1.96 * irf_se              # 95% CI lower
irf_upper = irf_vals + 1.96 * irf_se              # 95% CI upper
horizons  = np.arange(irf_vals.shape[0])           # 0 … 10

# Labels for the 2×2 grid
NAMES = ['BTC/USD', 'BTC/USDC']
PANEL_LABELS = [
    ['(A)', '(B)'],
    ['(C)', '(D)'],
]


def plot_irf(rc_dict, figsize_w, figsize_h, lw_main, lw_ci, save_dir, dpi=300):
    """
    2×2 grid of orthogonalized impulse-response functions.
    Row i = response of variable i; Column j = shock to variable j.
    """
    with plt.rc_context(rc_dict):
        fig, axes = plt.subplots(2, 2, figsize=(figsize_w, figsize_h),
                                  sharex=True)

        for i in range(2):       # response variable
            for j in range(2):   # shock variable
                ax = axes[i, j]
                resp = irf_vals[:, i, j]
                lo   = irf_lower[:, i, j]
                hi   = irf_upper[:, i, j]

                ax.fill_between(horizons, lo, hi,
                                color=CB_BLUE, alpha=0.18, linewidth=0)
                ax.plot(horizons, resp,
                        color=CB_BLUE, linewidth=lw_main, zorder=3)
                ax.axhline(0, color='black', linewidth=0.5,
                           linestyle='--', alpha=0.4)

                title = (f'{PANEL_LABELS[i][j]}  '
                         f'{NAMES[j]} $\\rightarrow$ {NAMES[i]}')
                ax.set_title(title)

                if i == 1:
                    ax.set_xlabel('Lag (minutes)')
                if j == 0:
                    ax.set_ylabel('Response (bps)')

                ax.set_xlim(0, 10)
                ax.xaxis.set_major_locator(mticker.MultipleLocator(2))

        fig.tight_layout(pad=0.5, h_pad=1.0, w_pad=1.0)
        path = os.path.join(save_dir, 'fig_var_irf.png')
        fig.savefig(path, dpi=dpi, bbox_inches='tight')
        plt.close(fig)
        print(f"  saved → {path}")


# Full-width version
plot_irf(FULL_RC, figsize_w=10, figsize_h=7, lw_main=1.6, lw_ci=0.8,
         save_dir=FIGURES_DIR)
# Column version
plot_irf(COL_RC, figsize_w=3.4, figsize_h=3.4, lw_main=0.9, lw_ci=0.5,
         save_dir=FIGURES_COL)

print("\nAll additional figures generated successfully.")


  saved → /Users/dhruvkohli/Desktop/Github Repos/IAQF2026/figures/fig_svb_crisis_zoom.png
  saved → /Users/dhruvkohli/Desktop/Github Repos/IAQF2026/figures_col/fig_svb_crisis_zoom.png

VAR: AIC-selected lags = 10, N = 26548
  saved → /Users/dhruvkohli/Desktop/Github Repos/IAQF2026/figures/fig_var_irf.png
  saved → /Users/dhruvkohli/Desktop/Github Repos/IAQF2026/figures_col/fig_var_irf.png

All additional figures generated successfully.


---
# Section 10: Verification Summary

Verify that all expected outputs have been generated.


In [24]:
# Verify all outputs
print('=== OUTPUT VERIFICATION ===')
print()

# Check figures
expected_figs = [
    'fig_stablecoin_peg.png', 'fig_dispersion_vs_adjusted_kraken.png',
    'fig_stablecoin_substitution_scatter.png', 'fig_half_life_robustness.png',
    'fig_two_layer_persistence.png', 'fig_svb_crisis_zoom.png',
    'fig_cross_exchange_basis.png', 'fig_liquidity_roll_amihud.png',
    'fig_volume_share.png', 'fig_realized_volatility.png',
    'fig_tail_blowout_kde.png', 'fig_correlation_regime_heatmap.png',
    'fig_var_irf.png', 'fig_arbitrage_after_fees.png',
]
print('Column Figures (figures_col/):')
for f in expected_figs:
    path = os.path.join(FIGURES_COL, f)
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  [{status}] {f}')

# Check tables
expected_tables = [
    'dispersion_adjusted_stats.csv', 'ou_basis_stats.csv', 'half_life_robustness.csv',
    'regression_hac.tex', 'cointegration_johansen.csv', 'price_discovery_metrics.csv',
    'granger_causality_fdr.csv', 'arbitrage_summary.csv', 'arbitrage_compact.tex',
    'data_coverage_core.csv', 'genius_counterfactual.csv',
    'hac_headline_metrics.csv', 'distributional_robustness.csv',
]
print('\nTables (tables/):')
for f in expected_tables:
    path = os.path.join(TABLES_DIR, f)
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  [{status}] {f}')

print('\n=== VERIFICATION COMPLETE ===')


=== OUTPUT VERIFICATION ===

Column Figures (figures_col/):
  [OK] fig_stablecoin_peg.png
  [OK] fig_dispersion_vs_adjusted_kraken.png
  [OK] fig_stablecoin_substitution_scatter.png
  [OK] fig_half_life_robustness.png
  [OK] fig_two_layer_persistence.png
  [OK] fig_svb_crisis_zoom.png
  [OK] fig_cross_exchange_basis.png
  [OK] fig_liquidity_roll_amihud.png
  [OK] fig_volume_share.png
  [OK] fig_realized_volatility.png
  [OK] fig_tail_blowout_kde.png
  [OK] fig_correlation_regime_heatmap.png
  [OK] fig_var_irf.png
  [OK] fig_arbitrage_after_fees.png

Tables (tables/):
  [OK] dispersion_adjusted_stats.csv
  [OK] ou_basis_stats.csv
  [OK] half_life_robustness.csv
  [OK] regression_hac.tex
  [OK] cointegration_johansen.csv
  [OK] price_discovery_metrics.csv
  [OK] granger_causality_fdr.csv
  [OK] arbitrage_summary.csv
  [OK] arbitrage_compact.tex
  [OK] data_coverage_core.csv
  [OK] genius_counterfactual.csv
  [OK] hac_headline_metrics.csv
  [OK] distributional_robustness.csv

=== VERIFICA